# CS-781 Phase 7 — Prior Case Retrieval · LegalLM

Complete standalone notebook: load Phase-1 weights → adapt a shared query/candidate encoder → train → evaluate → rank the entire test candidate pool → validate and zip `prediction.json`.

**Before Run All:**
1. Enable a single Kaggle T4 GPU and Internet (Internet is unnecessary with locally attached official PCR data).
2. In Section 1 set `CKPT_PATH` and `TOK_PATH` to your **trained Phase-1** assets. For Mamba this is your `mamba.pt` (or its actual saved filename), not a new randomly initialized backbone.
3. Select one `EXPERIMENT`. Set `LOCAL_DATA_DIR` or supply dataset access through the Kaggle secret named by `HF_SECRET_NAME` / `HF_TOKEN`. Never paste a token into the notebook.
4. Keep `RUN_PIPELINE=True` and run every cell. The last cell executes `RESULTS = main()`.
5. Download `/kaggle/working/submission_<EXPERIMENT>.zip`. Each notebook is independent; simultaneous runs need separate GPU sessions. Do not launch all three on a single T4.

**Scope and limitations:** all nine sections and declared presets are implemented. Source tokenizer/backbone cells below match your supplied reference exactly. Actual Phase-1 files and the authenticated PCR dataset were not available in the development runtime, and no T4 run or real score is claimed. Hyperparameters and throughput must be validated on your Kaggle run. The budget controller reserves full-pool indexing before allocating training; it can fail early if 90 minutes is insufficient. A wall-clock estimate cannot guarantee an absolute deadline under changing hardware/I/O speed.

**Rules:** no external pretrained weights or external datasets. Hugging Face Hub is used only to download the six official data files. BERTScore and text generation are not used for PCR.


## Section 1 · Configuration and experiment switch

Only change `EXPERIMENT` to select a technique. The checkpoint/tokenizer paths remain architecture-specific. Each run receives its own artifact directory. These are starting hyperparameters, not tuned values.

Mamba presets preserve `M1_adapter`, `M2_partial`, `M3_kd`, `M4_lorma`. The assignment lists full/partial FT, QLoRA, LORMA, prefix, prompt, IA3 and KD; adapters are supplemental. A six-technique plan across these notebooks is **D1_lorma, D2_qlora, D4_prompt, E1_full, E2_prefix, M3_kd**. Three runs produce three submissions, but do not by themselves complete a six-experiment requirement.


In [38]:
import os, sys, json, re, math, time, random, heapq, hashlib, copy, zipfile
from pathlib import Path
from collections import Counter, defaultdict
from dataclasses import dataclass, asdict
from contextlib import nullcontext
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.checkpoint

BACKBONE = "decoder"              # Fixed for this architecture-specific notebook.
EXPERIMENT = "D2_qlora"         # The only experiment selector.
SEED = 42
EOS_PLACEHOLDER = "<|endoftext|>"  # Exact constant used by the supplied reference tokenizer.

# Set these to YOUR TRAINED PHASE-1 assets. Section 3 will perform the actual load.
CKPT_PATH = "/kaggle/input/models/ashishkj93/phase-1-decoder-bundle/pytorch/default/1/best.pt"
TOK_PATH = "/kaggle/input/models/ashishkj93/phase-1-decoder-bundle/pytorch/default/1/bpe_merges.json"
EXPECTED_TOKENIZER_SHA256 = ""     # Optional known Phase-1 fingerprint; never infer from vocab size.
TRUST_PHASE1_PICKLE = True         # Matches your own torch.save dict; never use untrusted checkpoints.

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
HF_REVISION = "main"              # Pin the dataset commit when running final experiments.
HF_SECRET_NAME = "capstone-phase1-token"
LOCAL_DATA_DIR = ""               # Optional local directory containing Phase_7_PCR/Train, Dev, Test.
DATA_FILES = {
    "train_queries": "Phase_7_PCR/Train/train.json",
    "train_candidates": "Phase_7_PCR/Train/train_candidates.json",
    "dev_queries": "Phase_7_PCR/Dev/dev.json",
    "dev_candidates": "Phase_7_PCR/Dev/dev_candidates.json",
    "test_queries": "Phase_7_PCR/Test/test.json",
    "test_candidates": "Phase_7_PCR/Test/test_candidates.json",
}

@dataclass(frozen=True)
class Preset:
    method: str
    lr_backbone: float
    lr_head: float
    train_top: int = 0
    listed_in_phase7: bool = True

PRESETS = {
    'D1_lorma': Preset(method='lorma', lr_backbone=0.0002, lr_head=0.0005),
    'D2_qlora': Preset(method='qlora', lr_backbone=0.0002, lr_head=0.0005),
    'D3_kd': Preset(method='kd', lr_backbone=3e-05, lr_head=0.0003),
    'D4_prompt': Preset(method='prompt', lr_backbone=0.001, lr_head=0.0005),
    'D5_full': Preset(method='full', lr_backbone=2e-05, lr_head=0.0003),
    'D6_partial': Preset(method='partial', lr_backbone=5e-05, lr_head=0.0003, train_top=3),
    'D7_ia3': Preset(method='ia3', lr_backbone=0.0003, lr_head=0.0005),
    'D8_adapter': Preset(method='adapter', lr_backbone=0.0002, lr_head=0.0005, listed_in_phase7=False),
}
if EXPERIMENT not in PRESETS:
    raise ValueError(f"Unknown preset {EXPERIMENT!r}. Choose from {list(PRESETS)}")
EXP = PRESETS[EXPERIMENT]

# Adaptation settings: implemented in Section 5.
RANK, LORA_ALPHA = 8, 16.0
LORMA_SIDE = "out"                # W' = (I + (alpha/r) B A) W0; do not silently switch sides.
LOW_RANK_DROPOUT = 0.0            # Exact LORMA matrix identity, including during training.
ADAPTER_DIM, ADAPTER_DROPOUT = 64, 0.0
SOFT_PROMPT_TOKENS, KV_PREFIX_TOKENS = 32, 32
NF4_BLOCK_SIZE = 64               # Packed NF4 base codes plus per-block FP32 scales.
IA3_TARGETS = ("attention_k", "attention_v", "ffn_intermediate")
TRAIN_EMBEDDINGS_IN_PARTIAL = False
TRAIN_FINAL_NORM_IN_PARTIAL = True

# Shared dual encoder: same backbone and projection for query/candidate.
SHARE_QUERY_CANDIDATE_WEIGHTS = True
CHUNK_TOK = 1024                  # Content tokens/window; EOS/prompt reserve is added separately.
CHUNK_OVERLAP = 0
MAX_CHUNKS = 3                    # At most 4 separate windows, NOT a single 4096-token forward pass.
CHUNK_POLICY = "uniform"          # uniform | head_tail; preserve sentence boundaries when possible.
APPEND_EOS_PER_CHUNK = True
TOKEN_POOL = "mean"              # Later masked mean of content tokens, excluding PAD/prompt/EOS.
CHUNK_POOL = "mean"              # mean | attention; attention has a trainable pooling head.
EMBED_DIM, HEAD_DROPOUT = 256, 0.0
L2_NORMALIZE = True
SIMILARITY = "cosine"
INFO_NCE_TEMPERATURE = 0.07

# Candidate-set construction: deduplicate IDs, mark every included known positive.
CONTRASTIVE_BATCH_QUERIES = 2
POSITIVES_PER_QUERY = 1           # Sample at least one known positive; other included golds also count.
HARD_NEGATIVES_PER_QUERY = 1
RANDOM_NEGATIVES_PER_QUERY = 1
USE_IN_BATCH_NEGATIVES = True
MULTI_POSITIVE_LOSS = "positive_logsumexp"
HARD_NEGATIVE_SOURCE = "train_only_bm25_then_dense"
HARD_MINING_TOPK = 32
HARD_REFRESH_EPOCHS = 1
GRAD_ACCUM = 4                   # More optimizer accumulation does NOT enlarge an InfoNCE candidate set.
CHUNKS_PER_FORWARD = 1
GRAD_CKPT = False                # Original causal checkpoint branches drop padding masks.
USE_GRAD_CACHE = True            # Exact two-pass chunk-feature gradient replay in Section 5.

# KD: full-depth Phase-7 retrieval teacher trained from Phase 1, then frozen.
USE_KD = EXP.method == "kd"
TEACHER_POLICY = "train_then_cache"  # train_then_cache | load_phase7_teacher
TEACHER_RETRIEVAL_CKPT = ""          # Must contain our retrieval backbone + trained projection/pooler.
TEACHER_METHOD = "full"
TEACHER_BUDGET_FRACTION = 0.30
KD_ALPHA, KD_TEMPERATURE = 0.5, 2.0
STUDENT_LAYER_POLICY = "odd_indices"
KD_SIGNAL = "similarity_distribution"
KD_SCORE_DOMAIN = "cosine_over_info_nce_temperature"
KD_CANDIDATE_ALIGNMENT = "exact_candidate_id_order"

# Evaluation/indexing rules: official gold sets are never truncated to 7.
TOP_K = 7
PRIMARY_METRIC = "micro_f1_at_7"
SECONDARY_METRICS = ("micro_precision_at_7", "micro_recall_at_7")
STRICT_STRING_IDS = True          # Preserve leading zeros. Never strip zeros or cast through int.
EXCLUDE_SELF_MATCH = False        # No undocumented candidate exclusion.
TIE_BREAK = "candidate_id_lexicographic"
EMPTY_TRAIN_GOLD_POLICY = "skip_and_audit"
INVALID_GOLD_ID_POLICY = "error"  # Do not drop unknown gold IDs and inflate the score.
EMPTY_DOCUMENT_POLICY = "error"
EVAL_QUERY_BLOCK_SIZE, SCORE_CANDIDATE_BLOCK_SIZE = 32, 8192
INDEX_STORAGE_DTYPE = "float32"
SCORE_DTYPE = "float32"

# Section 7 will measure throughput; these are budgets, not measured timings.
EPOCHS = 3
TRAIN_QUERIES_PER_EPOCH = None
MAX_DEV_QUERIES = None           # Full dev evaluation, if budget permits; never cap the candidate pool.
AUTO_FIT_BUDGET = True
TOTAL_GPU_MINUTES = 120.0
BUDGET_SAFETY_FRACTION = 0.20
BENCHMARK_WARMUP_STEPS, BENCHMARK_STEPS = 2, 4
BENCHMARK_MARGIN = 1.25
CHECKPOINT_RESERVE_SECONDS = 60.0
WEIGHT_DECAY, CLIP_NORM, WARMUP_FRACTION = 0.01, 1.0, 0.06

OUTPUT_ROOT = Path("/kaggle/working") if Path("/kaggle").is_dir() else Path.cwd() / "outputs"
RUN_TAG = time.strftime("%Y%m%d-%H%M%S")
EXP_DIR = OUTPUT_ROOT / "phase7_pcr_decoder" / EXPERIMENT / RUN_TAG
BEST_CHECKPOINT_PATH = EXP_DIR / f"best_{EXPERIMENT}.pt"
RESULTS_PATH = EXP_DIR / f"results_{EXPERIMENT}.json"
PREDICTION_PATH = EXP_DIR / "prediction.json"
SUBMISSION_PATH = EXP_DIR / f"submission_{EXPERIMENT}.zip"
TEACHER_CACHE_DIR = EXP_DIR / "teacher_cache"
# main() creates the run directory and outputs.

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
AMP_DTYPE = torch.float16
PARAMETER_DTYPE = torch.float32
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.backends.cudnn.benchmark = False

def amp_context():
    return torch.autocast("cuda", dtype=torch.float16) if DEVICE.type == "cuda" else nullcontext()

def make_grad_scaler():
    return torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")

def validate_configuration():
    if TOP_K != 7: raise ValueError("Competition output requires exactly 7 candidates/query.")
    if not SHARE_QUERY_CANDIDATE_WEIGHTS: raise ValueError("This design uses one shared dual encoder.")
    if not 0 <= CHUNK_OVERLAP < CHUNK_TOK or MAX_CHUNKS < 1: raise ValueError("Invalid chunk settings.")
    if CHUNK_POLICY not in {"uniform", "head_tail"}: raise ValueError("Unsupported chunk policy.")
    if TOKEN_POOL != "mean" or CHUNK_POOL not in {"mean", "attention"}: raise ValueError("Unsupported pooling.")
    if EMBED_DIM <= 0 or not 0 <= HEAD_DROPOUT < 1: raise ValueError("Invalid retrieval head settings.")
    if not L2_NORMALIZE or SIMILARITY != "cosine": raise ValueError("Expected normalized cosine embeddings.")
    if INFO_NCE_TEMPERATURE <= 0 or KD_TEMPERATURE <= 0: raise ValueError("Temperature must be positive.")
    if RANK < 1 or LORA_ALPHA <= 0 or NF4_BLOCK_SIZE % 2: raise ValueError("Invalid low-rank/NF4 settings.")
    if LORMA_SIDE != "out" or LOW_RANK_DROPOUT != 0: raise ValueError("Preserve the requested exact LORMA formula.")
    if CONTRASTIVE_BATCH_QUERIES < 1 or POSITIVES_PER_QUERY < 1: raise ValueError("Need queries and positives.")
    if min(HARD_NEGATIVES_PER_QUERY, RANDOM_NEGATIVES_PER_QUERY) < 0: raise ValueError("Negative count must be >=0.")
    if CONTRASTIVE_BATCH_QUERIES == 1 and HARD_NEGATIVES_PER_QUERY + RANDOM_NEGATIVES_PER_QUERY == 0:
        raise ValueError("A single-query contrastive batch needs explicit negatives.")
    if min(EPOCHS, GRAD_ACCUM, CHUNKS_PER_FORWARD, BENCHMARK_STEPS) < 1: raise ValueError("Invalid batch/epoch settings.")
    if TOTAL_GPU_MINUTES <= 0 or not 0 <= BUDGET_SAFETY_FRACTION < 1: raise ValueError("Invalid time budget.")
    if TRAIN_QUERIES_PER_EPOCH is not None and TRAIN_QUERIES_PER_EPOCH < 1: raise ValueError("Invalid train cap.")
    if MAX_DEV_QUERIES is not None and MAX_DEV_QUERIES < 1: raise ValueError("Invalid dev cap.")
    if EXP.method == "partial" and EXP.train_top < 1: raise ValueError("Partial FT needs train_top.")
    if EXP.method == "prefix" and BACKBONE != "encdec": raise ValueError("Prefix preset uses encoder attention.")
    if not 0 <= KD_ALPHA <= 1 or not 0 < TEACHER_BUDGET_FRACTION < 1: raise ValueError("Invalid KD settings.")
    if TEACHER_POLICY not in {"train_then_cache", "load_phase7_teacher"}: raise ValueError("Invalid teacher policy.")
    if USE_KD and TEACHER_POLICY == "load_phase7_teacher" and not TEACHER_RETRIEVAL_CKPT:
        raise ValueError("Set TEACHER_RETRIEVAL_CKPT or use train_then_cache.")
    return True

validate_configuration()
CONFIG_SNAPSHOT = {
    "phase": 7, "backbone": BACKBONE, "experiment": EXPERIMENT, "preset": asdict(EXP),
    "checkpoint_path": CKPT_PATH, "tokenizer_path": TOK_PATH,
    "data_files": DATA_FILES, "seed": SEED, "chunk_tokens": CHUNK_TOK, "max_chunks": MAX_CHUNKS,
    "pooling": [TOKEN_POOL, CHUNK_POOL], "embedding_dim": EMBED_DIM, "top_k": TOP_K,
    "total_gpu_minutes": TOTAL_GPU_MINUTES, "scope": "complete_phase7",
}
print("Phase 7 |", BACKBONE, "|", EXPERIMENT, "|", asdict(EXP))
print("PyTorch:", torch.__version__, "| device:", DEVICE, "| AMP:", "FP16" if DEVICE.type == "cuda" else "off (FP32)")
if DEVICE.type == "cuda": print("GPU:", torch.cuda.get_device_name(0))
print("Phase-1 checkpoint configured:", CKPT_PATH)
print("Planned output:", EXP_DIR)
if not EXP.listed_in_phase7:
    print("SUPPLEMENTAL: adapters are requested here but not in the PDF's Phase-7 technique list.")
print("Configuration selected. The final cell loads Phase 1, trains PCR and creates the submission.")

# Complete pipeline controls. Set paths above before running the final cell.
RUN_PIPELINE = True
ALLOW_CPU_PIPELINE = False        # Only enable for small synthetic development fixtures.
RETRIEVAL_GRAD_CKPT = True        # Correct per-block replay in the retrieval wrapper.
INFERENCE_BENCHMARK_DOCS = 8
STUDENT_BENCHMARK_RESERVE_SECONDS = 120.0
LOG_EVERY_UPDATES = 10
TOKEN_MEMORY_CACHE_DOCS = 256


Phase 7 | decoder | D2_qlora | {'method': 'qlora', 'lr_backbone': 0.0002, 'lr_head': 0.0005, 'train_top': 0, 'listed_in_phase7': True}
PyTorch: 2.11.0+cu130 | device: cuda:0 | AMP: FP16
GPU: Tesla T4
Phase-1 checkpoint configured: /kaggle/input/models/ashishkj93/phase-1-decoder-bundle/pytorch/default/1/best.pt
Planned output: /kaggle/working/phase7_pcr_decoder/D2_qlora/20261108-164418
Configuration selected. The final cell loads Phase 1, trains PCR and creates the submission.


## Section 2 · Exact Phase-1 tokenizer and architecture

The following two code cells are copied verbatim from the matching Phase-5 reference. All retrieval changes are in separate wrappers after strict checkpoint loading.

In [39]:
CHAR_MAP = str.maketrans({
    "\u2018": "'", "\u2019": "'", "\u201c": '"', "\u201d": '"', "\u2013": "-", "\u2014": "-",
    "\u2212": "-", "\u2026": "...", "\u00a0": " ", "\u200b": "", "\ufeff": "", "\u00ad": "",
})
CTRL = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]")

def normalize_text(t: str, keep_newlines: bool) -> str:
    """Character and whitespace normalisation. Applied to train text AND test prompts."""
    t = CTRL.sub(" ", t.translate(CHAR_MAP))
    if keep_newlines:
        t = re.sub(r"[ \t]+", " ", t)
        t = re.sub(r" ?\n ?", "\n", t)
        t = re.sub(r"\n{3,}", "\n\n", t)
    else:
        t = re.sub(r"\s+", " ", t)
    return t.strip()

PRETOKEN_RE = re.compile(r"""'(?:[sdmt]|ll|ve|re)| ?[^\W\d_]+| ?\d{1,3}| ?(?:[^\s\w]|_)+|\s+(?!\S)|\s+""")

class ByteBPE:
    def __init__(self, merges=None):
        self.merges = [tuple(m) for m in (merges or [])]
        self.rank = {p: i for i, p in enumerate(self.merges)}
        self.vocab = [bytes([i]) for i in range(256)]
        for a, b in self.merges:
            self.vocab.append(self.vocab[a] + self.vocab[b])
        self.eos_id = len(self.vocab)
        self.vocab.append(EOS_PLACEHOLDER.encode())
        self.vocab_size = len(self.vocab)
        self._cache = {}

    # ---------- training ----------
    @staticmethod
    def train(chunk_counts, vocab_size, min_freq=3, log_every=2000):
        n_merges = vocab_size - 256 - 1                      # last id is EOS
        items = [(c.encode("utf-8"), f) for c, f in chunk_counts.items() if f >= min_freq]
        words = [list(b) for b, _ in items]
        freqs = [f for _, f in items]
        pair_cnt, where = defaultdict(int), defaultdict(set)
        for i, w in enumerate(words):
            for p in zip(w, w[1:]):
                pair_cnt[p] += freqs[i]; where[p].add(i)
        heap = [(-c, p) for p, c in pair_cnt.items()]
        heapq.heapify(heap)
        merges, t0 = [], time.time()
        while len(merges) < n_merges:
            best = None
            while heap:                                       # lazy deletion
                negc, p = heapq.heappop(heap)
                c = pair_cnt.get(p, 0)
                if c <= 0: continue
                if c != -negc:
                    heapq.heappush(heap, (-c, p)); continue
                best = p; break
            if best is None: break
            a, b = best; new_id = 256 + len(merges); merges.append(best)
            changed = set()
            for i in list(where[best]):
                w, f = words[i], freqs[i]
                out, j, n, hit = [], 0, len(w), False
                while j < n:
                    if j < n - 1 and w[j] == a and w[j + 1] == b:
                        out.append(new_id); j += 2; hit = True
                    else:
                        out.append(w[j]); j += 1
                if not hit: continue
                for q in zip(w, w[1:]): pair_cnt[q] -= f
                for q in zip(out, out[1:]):
                    pair_cnt[q] += f; where[q].add(i); changed.add(q)
                words[i] = out
            where.pop(best, None); pair_cnt.pop(best, None)
            for q in changed:
                if pair_cnt[q] > 0: heapq.heappush(heap, (-pair_cnt[q], q))
            if len(merges) % log_every == 0:
                print(f"  merges {len(merges):6d}/{n_merges} | {time.time() - t0:5.0f}s")
        return ByteBPE(merges)

    # ---------- encoding / decoding ----------
    def _encode_chunk(self, chunk):
        hit = self._cache.get(chunk)
        if hit is not None: return hit
        ids = list(chunk.encode("utf-8"))
        while len(ids) > 1:
            best_r, best_p = None, None
            for p in zip(ids, ids[1:]):
                r = self.rank.get(p)
                if r is not None and (best_r is None or r < best_r):
                    best_r, best_p = r, p
            if best_p is None: break
            a, b = best_p; new_id = 256 + best_r; out, j = [], 0
            while j < len(ids):
                if j < len(ids) - 1 and ids[j] == a and ids[j + 1] == b:
                    out.append(new_id); j += 2
                else:
                    out.append(ids[j]); j += 1
            ids = out
        if len(self._cache) > 3_000_000: self._cache.clear()
        self._cache[chunk] = ids
        return ids

    def encode(self, text):
        ids = []
        for chunk in PRETOKEN_RE.findall(text):
            ids.extend(self._encode_chunk(chunk))
        return ids

    def decode(self, ids):
        data = b"".join(self.vocab[i] for i in ids if i != self.eos_id)
        return data.decode("utf-8", errors="replace")

    def save(self, path):
        json.dump({"merges": self.merges, "eos": EOS_PLACEHOLDER}, open(path, "w"))

    @staticmethod
    def load(path):
        return ByteBPE(json.load(open(path))["merges"])

In [40]:
@dataclass
class ModelConfig:
    vocab_size: int
    n_ctx: int = 1280
    n_layer: int = 12
    n_head: int = 12
    n_kv_head: int = 4
    n_embd: int = 768
    n_ffn: int = 2816
    rope_base: float = 10000.0

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim)); self.eps = eps
    def forward(self, x):
        xf = x.float()
        return (xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)).type_as(x) * self.weight

def build_rope(head_dim, n_ctx, base):
    inv = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
    freqs = torch.outer(torch.arange(n_ctx).float(), inv)
    return freqs.cos(), freqs.sin()                          # (n_ctx, head_dim/2)

def apply_rope(x, cos, sin):                                 # x: (B, H, T, hd); cos/sin: (*, 1, T, hd/2)
    x1, x2 = x.float().chunk(2, dim=-1)
    return torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).type_as(x)

class Attention(nn.Module):
    def __init__(self, c: ModelConfig):
        super().__init__()
        self.nh, self.nkv, self.hd = c.n_head, c.n_kv_head, c.n_embd // c.n_head
        assert c.n_head % c.n_kv_head == 0
        self.q = nn.Linear(c.n_embd, self.nh * self.hd, bias=False)
        self.k = nn.Linear(c.n_embd, self.nkv * self.hd, bias=False)
        self.v = nn.Linear(c.n_embd, self.nkv * self.hd, bias=False)
        self.o = nn.Linear(self.nh * self.hd, c.n_embd, bias=False)
    def forward(self, x, cos, sin, mask=None, cache=None):
        B, T, _ = x.shape
        q = self.q(x).view(B, T, self.nh, self.hd).transpose(1, 2)
        k = self.k(x).view(B, T, self.nkv, self.hd).transpose(1, 2)
        v = self.v(x).view(B, T, self.nkv, self.hd).transpose(1, 2)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        new_cache = None
        if cache is not None:                                # generation: grow the KV-cache
            if cache[0] is not None:
                k, v = torch.cat([cache[0], k], 2), torch.cat([cache[1], v], 2)
            new_cache = (k, v)
        rep = self.nh // self.nkv
        if rep > 1:
            k, v = k.repeat_interleave(rep, 1), v.repeat_interleave(rep, 1)
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask,
                                           is_causal=(mask is None and cache is None))
        return self.o(y.transpose(1, 2).reshape(B, T, -1)), new_cache

class SwiGLU(nn.Module):
    def __init__(self, c: ModelConfig):
        super().__init__()
        self.gate = nn.Linear(c.n_embd, c.n_ffn, bias=False)
        self.up = nn.Linear(c.n_embd, c.n_ffn, bias=False)
        self.down = nn.Linear(c.n_ffn, c.n_embd, bias=False)
    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))

class Block(nn.Module):
    def __init__(self, c: ModelConfig):
        super().__init__()
        self.n1, self.attn = RMSNorm(c.n_embd), Attention(c)
        self.n2, self.mlp = RMSNorm(c.n_embd), SwiGLU(c)
    def forward(self, x, cos, sin, mask=None, cache=None):
        h, cache = self.attn(self.n1(x), cos, sin, mask, cache)
        x = x + h
        return x + self.mlp(self.n2(x)), cache

class LegalLM(nn.Module):
    def __init__(self, c: ModelConfig):
        super().__init__()
        self.c = c
        self.tok_emb = nn.Embedding(c.vocab_size, c.n_embd)
        self.blocks = nn.ModuleList([Block(c) for _ in range(c.n_layer)])
        self.norm = RMSNorm(c.n_embd)
        self.head = nn.Linear(c.n_embd, c.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight               # weight tying
        cos, sin = build_rope(c.n_embd // c.n_head, c.n_ctx, c.rope_base)
        self.register_buffer("cos", cos, persistent=False)
        self.register_buffer("sin", sin, persistent=False)
        self.grad_ckpt = False
        self.apply(self._init)
        for n, p in self.named_parameters():                 # GPT-2 style scaled residual init
            if n.endswith("attn.o.weight") or n.endswith("mlp.down.weight"):
                nn.init.normal_(p, 0.0, 0.02 / math.sqrt(2 * c.n_layer))
    @staticmethod
    def _init(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, 0.0, 0.02)

    def forward(self, idx, targets=None, pos_ids=None, mask=None, caches=None, last_only=False):
        B, T = idx.shape
        x = self.tok_emb(idx)
        if pos_ids is None:
            cos, sin = self.cos[:T][None, None], self.sin[:T][None, None]
        else:                                                # left-padded batches carry their own positions
            cos, sin = self.cos[pos_ids][:, None], self.sin[pos_ids][:, None]
        new_caches = [] if caches is not None else None
        for i, blk in enumerate(self.blocks):
            if self.grad_ckpt and self.training:
                x = torch.utils.checkpoint.checkpoint(
                    lambda x_, c_, s_, b=blk: b(x_, c_, s_)[0], x, cos, sin, use_reentrant=False)
            else:
                x, c_new = blk(x, cos, sin, mask, caches[i] if caches is not None else None)
                if new_caches is not None: new_caches.append(c_new)
        x = self.norm(x)
        if last_only: x = x[:, -1:]
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss, new_caches

In [41]:
PHASE1_CONFIG_CLASS = ModelConfig
PHASE1_MODEL_CLASS = LegalLM

def embedding_layer(model):
    return model.emb if BACKBONE == "encdec" else model.tok_emb

def retrieval_layers(model):
    return model.enc_blocks if BACKBONE == "encdec" else model.blocks

def retrieval_final_norm(model):
    return model.enc_norm if BACKBONE == "encdec" else model.norm

def hidden_size(cfg):
    return cfg.n_embd if BACKBONE == "decoder" else cfg.d_model

def validate_checkpoint_config(cfg, tokenizer=None, preset=None):
    """Pure validation for Section 3; does not instantiate or load a model."""
    preset = EXP if preset is None else preset
    positions = CHUNK_TOK + int(APPEND_EOS_PER_CHUNK)
    if preset.method == "prompt": positions += SOFT_PROMPT_TOKENS
    # Encoder K/V prefixes do not advance document RoPE positions in the planned implementation.
    if positions > cfg.n_ctx:
        raise ValueError(f"Content + EOS + soft prompt = {positions} exceeds checkpoint context {cfg.n_ctx}.")
    dim = hidden_size(cfg)
    if dim % cfg.n_head or (dim // cfg.n_head) % 2:
        raise ValueError("Invalid attention/RoPE dimensions.")
    if hasattr(cfg, "n_kv_head") and cfg.n_head % cfg.n_kv_head:
        raise ValueError("Query-head count must be divisible by KV-head count.")
    depth = cfg.n_enc if BACKBONE == "encdec" else cfg.n_layer if BACKBONE == "decoder" else len(cfg.pattern)
    if depth < 1: raise ValueError("Backbone has no retrieval layers.")
    if preset.method == "partial" and preset.train_top > depth: raise ValueError("train_top exceeds depth.")
    if preset.method == "kd" and depth < 2: raise ValueError("Half-depth KD requires at least 2 retrieval layers.")
    if BACKBONE == "mamba":
        if not cfg.pattern or not set(cfg.pattern) <= {"M", "A"}: raise ValueError("Invalid hybrid pattern.")
        if (cfg.expand * cfg.d_model) % cfg.headdim: raise ValueError("Invalid Mamba head dimensions.")
    if tokenizer is not None:
        gap = cfg.vocab_size - tokenizer.vocab_size
        if (BACKBONE != "encdec" and gap != 0) or (BACKBONE == "encdec" and not 0 <= gap <= 256):
            raise ValueError("Tokenizer/checkpoint vocabulary sizes disagree.")
        if not 0 <= tokenizer.eos_id < cfg.vocab_size: raise ValueError("EOS is outside the checkpoint vocabulary.")
    return {"retrieval_depth": depth, "hidden_size": dim, "checkpoint_context": cfg.n_ctx,
            "positions_per_window": positions, "uses_encoder_only": BACKBONE == "encdec"}

print("Exact backbone definitions available:", PHASE1_CONFIG_CLASS.__name__, PHASE1_MODEL_CLASS.__name__)


Exact backbone definitions available: ModelConfig LegalLM


### Tiny definition check
This explicit random CPU fixture verifies definitions only. It is discarded and is never used for the actual pipeline.

In [42]:
RUN_DEFINITION_TEST = True

def run_definition_test():
    """Tiny random CPU fixture only. Does not load or modify any user checkpoint."""
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(123)
        tok = ByteBPE([])
        sample = normalize_text("Court’s order\nwas upheld—today.", False)
        assert tok.decode(tok.encode(sample)) == sample
        assert chr(10) not in sample
        if BACKBONE == "decoder":
            cfg = ModelConfig(vocab_size=257, n_ctx=32, n_layer=2, n_head=4, n_kv_head=2,
                              n_embd=32, n_ffn=64)
        elif BACKBONE == "encdec":
            cfg = EncDecConfig(vocab_size=272, n_ctx=32, n_enc=2, n_dec=2,
                               d_model=32, n_head=4, n_ffn=64)
        else:
            cfg = HybridConfig(vocab_size=257, n_ctx=32, pattern="MMMA", d_model=32,
                               n_head=4, n_kv_head=2, n_ffn=64, expand=2, headdim=8,
                               d_state=8, d_conv=4, chunk=4)
        model = PHASE1_MODEL_CLASS(cfg).cpu().eval()
        clone = PHASE1_MODEL_CLASS(copy.deepcopy(cfg)).cpu().eval()
        clone.load_state_dict(model.state_dict(), strict=True)
        assert model.head.weight is embedding_layer(model).weight
        ids = torch.randint(0,256,(1,11))
        if BACKBONE == "encdec":
            hidden = model.encode(ids)
            torch.testing.assert_close(hidden, clone.encode(ids))
        else:
            # Exercise original blocks to get pre-head features, as retrieval will require.
            def features(m, tokens):
                x = m.tok_emb(tokens); length = tokens.size(1)
                cos, sin = m.cos[:length][None,None], m.sin[:length][None,None]
                for layer in m.blocks:
                    if BACKBONE == "mamba": x, _ = layer(x,cos,sin,None,None,False)
                    else: x, _ = layer(x,cos,sin)
                return m.norm(x)
            hidden = features(model,ids)
            torch.testing.assert_close(hidden,features(clone,ids))
            torch.testing.assert_close(model.head(hidden),model(ids)[0])
            changed = ids.clone(); changed[:,7:] = (changed[:,7:]+1) % 256
            torch.testing.assert_close(hidden[:,:7],features(model,changed)[:,:7],atol=3e-5,rtol=3e-4)
        assert hidden.shape == (1,11,32) and torch.isfinite(hidden).all()
        # A nonconstant loss on pooled features must reach retrieval backbone parameters.
        probe = nn.Linear(32,8)
        pooled = hidden.mean(1)
        loss = (probe(pooled) - torch.ones(1,8)).square().mean()
        loss.backward()
        grad = embedding_layer(model).weight.grad
        assert grad is not None and torch.isfinite(grad).all() and grad.abs().sum() > 0
        # Decoder blocks remain present at definition stage but are unused by encdec retrieval.
        if BACKBONE == "encdec":
            assert all(p.grad is None for p in model.dec_blocks.parameters())
        print("PASS: tokenizer round-trip; strict synthetic state load; tied weights; "
              "hidden-feature shape; finite retrieval-feature gradients." +
              (" Causal prefix invariance passed." if BACKBONE != "encdec" else " Encoder-only gradients passed."))

if RUN_DEFINITION_TEST:
    run_definition_test()


PASS: tokenizer round-trip; strict synthetic state load; tied weights; hidden-feature shape; finite retrieval-feature gradients. Causal prefix invariance passed.


## Section 3 · Checkpoint loading and verification

`load_phase1()` is called at the start of `main()`. It reads **your configured `.pt`**, constructs the architecture from `ck['cfg']`, and performs `load_state_dict(ck['model'], strict=True)` before any PEFT or KD pruning. It prints the checkpoint path, configuration, parameter count, step, and tokenizer fingerprint. Matching vocabulary size alone cannot establish matching BPE merges; use the original merges and optionally set the expected SHA256.

For encoder–decoder retrieval, the **full** Phase-1 model is verified first; only afterward are the unused decoder and vocabulary head removed. The full definitions remain untouched. The hybrid depth comes from its checkpoint's `pattern`, so the class default does not replace your trained `MMMA × 5` layout.


In [43]:
# Section 3 — strict Phase-1 checkpoint loading, BEFORE adaptation or pruning.
import gc
from dataclasses import fields

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''): h.update(block)
    return h.hexdigest()

def atomic_json(value, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    with open(tmp, 'w', encoding='utf-8') as f:
        json.dump(value, f, ensure_ascii=False, indent=2, allow_nan=False)
    os.replace(tmp, path)

def sync_device():
    if DEVICE.type == 'cuda': torch.cuda.synchronize()

def load_phase1():
    for p in (CKPT_PATH, TOK_PATH):
        if not Path(p).is_file():
            raise FileNotFoundError(f'Set CKPT_PATH/TOK_PATH in Section 1. Missing: {p}')
    ck = torch.load(CKPT_PATH, map_location='cpu', weights_only=not TRUST_PHASE1_PICKLE)
    if not isinstance(ck, dict) or not {'cfg', 'model'} <= ck.keys():
        raise ValueError('Expected the original Phase-1 checkpoint with cfg and model keys.')
    cfg_dict = asdict(ck['cfg']) if hasattr(ck['cfg'], '__dataclass_fields__') else dict(ck['cfg'])
    # Do not guess a model shape or silently ignore architecture config fields.
    cfg = PHASE1_CONFIG_CLASS(**cfg_dict)
    tok = ByteBPE.load(TOK_PATH)
    tok_sha = sha256_file(TOK_PATH)
    if EXPECTED_TOKENIZER_SHA256 and tok_sha != EXPECTED_TOKENIZER_SHA256:
        raise ValueError('Tokenizer fingerprint differs from the configured Phase-1 fingerprint.')
    verification = validate_checkpoint_config(cfg, tok)
    base = PHASE1_MODEL_CLASS(cfg)
    base.load_state_dict(ck['model'], strict=True)
    if base.head.weight is not embedding_layer(base).weight:
        raise RuntimeError('Phase-1 weight tying was lost.')
    parameter_count = sum(p.numel() for p in base.parameters())
    meta = dict(checkpoint=str(CKPT_PATH), checkpoint_sha256=sha256_file(CKPT_PATH),
                tokenizer_sha256=tok_sha, cfg=cfg_dict, phase1_parameters=parameter_count,
                checkpoint_step=str(ck.get('step', ck.get('global_step', 'not stored'))),
                verification=verification)
    print('LOADED PHASE-1 WEIGHTS:', CKPT_PATH, '| strict=True')
    print('Tokenizer:', TOK_PATH, '| SHA256:', tok_sha)
    print('Checkpoint configuration:', cfg_dict)
    print('Phase-1 parameters:', f'{parameter_count:,}', '| step:', meta['checkpoint_step'])
    # Full checkpoint has now been verified. Remove only modules unused by retrieval.
    base.head = nn.Identity()
    if BACKBONE == 'encdec':
        base.dec_blocks = nn.ModuleList(); base.dec_norm = nn.Identity()
    base.grad_ckpt = False  # Retrieval wrapper implements its own correct checkpoint path.
    del ck; gc.collect()
    return base, tok, meta


## Section 4 · Data, sentence flattening, chunking and train-only hard negatives

Each split uses its own candidate file. Paths resolve exactly first, with a unique filename fallback inside `Phase_7_PCR`. String IDs are preserved. Integer IDs require explicitly setting `STRICT_STRING_IDS=False`; output IDs are still strings. Duplicate document IDs, unknown gold IDs and empty documents fail with a clear error. Training queries with no gold positives are skipped and listed in `data_audit.json`; dev gold sets are retained intact.

Sentence-aware windows contain at most 1024 content tokens. Long sentences are split; uniformly selected windows (or head/tail windows) are pooled at case level. The default retains at most four windows, so this is a context-selection approximation. Disk token caches include the tokenizer fingerprint and all chunk settings. BM25 and subsequent dense hard-negative mining use only training text and training gold labels.


In [44]:
# Section 4 — official data only, split-specific candidate pools, bounded token windows.
def dataset_token():
    token = os.environ.get('HF_TOKEN')
    if not token:
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret(HF_SECRET_NAME)
        except Exception:
            pass
    return token  # Never print or write tokens to results.

def resolve_data_paths():
    if LOCAL_DATA_DIR:
        root = Path(LOCAL_DATA_DIR)
        paths = {}
        for key, name in DATA_FILES.items():
            exact = root / name
            hits = [exact] if exact.is_file() else sorted(root.rglob(Path(name).name))
            if len(hits) != 1:
                raise ValueError(f'Expected one local {Path(name).name}; found {hits}')
            paths[key] = str(hits[0])
        return paths, 'local'
    try:
        from huggingface_hub import HfApi, hf_hub_download
    except ImportError as exc:
        raise ImportError('Install huggingface_hub, or set LOCAL_DATA_DIR to the official attached data.') from exc
    token = dataset_token(); api = HfApi(token=token)
    info = api.dataset_info(HF_DATASET_REPO, revision=HF_REVISION)
    revision = info.sha  # Resolve main once; all six files use the same commit.
    names = api.list_repo_files(HF_DATASET_REPO, repo_type='dataset', revision=revision)
    paths = {}
    for key, desired in DATA_FILES.items():
        hits = [desired] if desired in names else [n for n in names
            if 'phase_7_pcr' in n.lower().split('/') and Path(n).name == Path(desired).name]
        if len(hits) != 1:
            raise ValueError(f'Expected one Phase_7_PCR/{Path(desired).name}; found {hits}')
        paths[key] = hf_hub_download(HF_DATASET_REPO, hits[0], repo_type='dataset',
                                    revision=revision, token=token)
    return paths, revision

def canonical_id(value):
    # Strings are kept EXACTLY: do not strip leading zeros or whitespace from IDs.
    if isinstance(value, str) and value: return value
    if not STRICT_STRING_IDS and isinstance(value, int) and not isinstance(value, bool): return str(value)
    raise ValueError(f'Expected a nonempty string ID; received {value!r}. '
                     'If official IDs are integers, set STRICT_STRING_IDS=False to convert them to strings.')

def read_cases(path, labelled=False):
    with open(path, encoding='utf-8') as f: raw = json.load(f)
    if isinstance(raw, dict):
        if set(raw) == {'data'} and isinstance(raw['data'], list): raw = raw['data']
        elif all(isinstance(v, dict) for v in raw.values()):
            rows = []
            for key, value in raw.items():
                row = dict(value)
                if 'id' in row and canonical_id(row['id']) != canonical_id(key):
                    raise ValueError('Outer and inner case IDs disagree.')
                row.setdefault('id', key); rows.append(row)
            raw = rows
    if not isinstance(raw, list): raise ValueError(f'{path}: expected a list of case objects.')
    records = {}; audit = {'blank_sentences': 0, 'duplicate_gold_ids': 0}
    for row in raw:
        if not isinstance(row, dict) or not {'id','text'} <= row.keys():
            raise ValueError(f'{path}: each case needs id and text.')
        key = canonical_id(row['id'])
        if key in records: raise ValueError(f'{path}: duplicate case ID {key!r}')
        sentences = row['text']
        if not isinstance(sentences, list) or not all(isinstance(s,str) for s in sentences):
            raise ValueError(f'{key}: text must be a list of sentence strings.')
        cleaned = [normalize_text(s, False) for s in sentences]
        audit['blank_sentences'] += sum(not s for s in cleaned)
        cleaned = [s for s in cleaned if s]
        if not cleaned: raise ValueError(f'{key}: empty document; correct the source data.')
        text = normalize_text(' '.join(cleaned), False)
        record = {'id':key, 'text':text, 'sentences':cleaned,
                  'doc_hash':hashlib.sha256(text.encode()).hexdigest()}
        if labelled:
            gold = row.get('relevant_candidates')
            if not isinstance(gold, list): raise ValueError(f'{key}: missing relevant_candidates list.')
            ids = [canonical_id(g) for g in gold]
            audit['duplicate_gold_ids'] += len(ids) - len(set(ids))
            record['gold'] = set(ids)  # Keep the complete official gold set, even if >7.
        records[key] = record
    if not records: raise ValueError(f'{path}: no cases.')
    return records, audit

def load_data():
    paths, revision = resolve_data_paths(); data = {}; audit = {}
    for key, path in paths.items():
        data[key], audit[key] = read_cases(path, key in {'train_queries','dev_queries'})
    for split in ('train','dev','test'):
        qs, cs = data[f'{split}_queries'], data[f'{split}_candidates']
        if len(cs) < TOP_K: raise ValueError(f'{split}: fewer than {TOP_K} candidates.')
        if split != 'test':
            for q in qs.values():
                unknown = q['gold'] - cs.keys()
                if unknown: raise ValueError(f'{split}/{q["id"]}: unknown gold IDs: {sorted(unknown)[:10]}')
        print(f'{split}: {len(qs)} queries; {len(cs)} candidates')
    skipped = [q['id'] for q in data['train_queries'].values() if not q['gold']]
    data['train_queries'] = {k:v for k,v in data['train_queries'].items() if v['gold']}
    if not data['train_queries']: raise ValueError('No training query has a positive candidate.')
    audit['skipped_train_empty_gold'] = skipped
    audit['data_revision'] = revision
    audit['file_sha256'] = {k:sha256_file(p) for k,p in paths.items()}
    atomic_json(audit, EXP_DIR/'data_audit.json')
    return data, audit

class TokenStore:
    """Sentence-aware windows; a long sentence is split at the token limit. Disk cached."""
    def __init__(self, tokenizer, tokenizer_sha):
        self.tok = tokenizer
        settings = [tokenizer_sha, CHUNK_TOK, CHUNK_OVERLAP, MAX_CHUNKS, CHUNK_POLICY,
                    APPEND_EOS_PER_CHUNK, 'sentence_chunks_v1']
        self.signature = hashlib.sha256(json.dumps(settings).encode()).hexdigest()
        self.root = OUTPUT_ROOT/'phase7_token_cache'/self.signature[:20]
        self.root.mkdir(parents=True, exist_ok=True)
        from collections import OrderedDict
        self.memo = OrderedDict()
    def __getitem__(self, record):
        key = hashlib.sha256(json.dumps(record['sentences'],ensure_ascii=False).encode()).hexdigest()
        if key in self.memo:
            self.memo.move_to_end(key);return self.memo[key]
        path = self.root/(key+'.json')
        if path.exists():
            with open(path) as f: chunks = json.load(f)
        else:
            chunks = []; current = []
            for i, sentence in enumerate(record['sentences']):
                tokens = self.tok.encode((' ' if i else '')+sentence)
                if len(tokens) > CHUNK_TOK:
                    if current: chunks.append(current); current = []
                    stride = CHUNK_TOK-CHUNK_OVERLAP
                    for start in range(0,len(tokens),stride):
                        part = tokens[start:start+CHUNK_TOK]
                        if part: chunks.append(part)
                        if start+CHUNK_TOK >= len(tokens): break
                elif len(current)+len(tokens) > CHUNK_TOK:
                    chunks.append(current)
                    overlap = current[-CHUNK_OVERLAP:] if CHUNK_OVERLAP else []
                    current = overlap+tokens if len(overlap)+len(tokens)<=CHUNK_TOK else tokens
                else: current.extend(tokens)
            if current: chunks.append(current)
            if not chunks: raise ValueError(f'No tokens for {record["id"]}')
            if len(chunks)>MAX_CHUNKS:
                if CHUNK_POLICY == 'uniform':
                    indices = np.linspace(0,len(chunks)-1,MAX_CHUNKS).round().astype(int).tolist()
                else:
                    head=(MAX_CHUNKS+1)//2; tail=MAX_CHUNKS-head
                    indices = list(range(head)) + (list(range(len(chunks)-tail,len(chunks))) if tail else [])
                chunks = [chunks[i] for i in indices]
            atomic_json(chunks,path)
        if any(not c or len(c)>CHUNK_TOK for c in chunks): raise ValueError('Invalid token cache.')
        self.memo[key] = chunks
        while len(self.memo)>TOKEN_MEMORY_CACHE_DOCS: self.memo.popitem(last=False)
        return chunks

def prepare_tokens(data, store):
    start = time.monotonic()
    for records in data.values():
        for i,record in enumerate(records.values()):
            store[record]
            if (i+1)%1000==0: print('Tokenized',i+1,'documents in current split/pool',flush=True)
    print('Tokenization/cache seconds:',round(time.monotonic()-start,2))

class BM25Miner:
    """Inverted-index BM25 fitted ONLY to train_candidates. No pretrained encoder."""
    def __init__(self, candidates):
        self.ids = sorted(candidates); self.postings = defaultdict(list)
        self.lengths = []
        for i,cid in enumerate(self.ids):
            terms = re.findall(r'\w+',candidates[cid]['text'].lower())
            self.lengths.append(len(terms))
            for term,tf in Counter(terms).items(): self.postings[term].append((i,tf))
        self.avg = max(1.,sum(self.lengths)/len(self.lengths))
    def retrieve(self, record, k):
        scores = defaultdict(float); n=len(self.ids)
        for word in set(re.findall(r'\w+',record['text'].lower())):
            postings=self.postings.get(word,()); df=len(postings)
            idf=math.log(1+(n-df+0.5)/(df+0.5))
            for i,tf in postings:
                scores[i]+=idf*tf*2.5/(tf+1.5*(0.25+0.75*self.lengths[i]/self.avg))
        ordered=sorted(scores,key=lambda i:(-scores[i],self.ids[i]))
        gold=record['gold']; result=[self.ids[i] for i in ordered if self.ids[i] not in gold][:k]
        if len(result)<k:
            seen=set(result)
            result.extend(cid for cid in self.ids if cid not in gold and cid not in seen)
        return result[:k]

def initial_hard_negatives(data):
    miner=BM25Miner(data['train_candidates'])
    result={qid:miner.retrieve(q,HARD_MINING_TOPK) for qid,q in data['train_queries'].items()}
    del miner
    return result

def sample_batch(query_ids, queries, candidates, hard, rng):
    cids=[]; all_ids=sorted(candidates)
    for qid in query_ids:
        gold=queries[qid]['gold']; positives=sorted(gold)
        cids.extend(rng.sample(positives,min(POSITIVES_PER_QUERY,len(positives))))
        available=[x for x in hard.get(qid,[]) if x not in gold]
        cids.extend(rng.sample(available,min(HARD_NEGATIVES_PER_QUERY,len(available))))
        # Rejection sampling is bounded; fallback handles tiny candidate pools.
        negatives=[]
        for _ in range(max(20,10*RANDOM_NEGATIVES_PER_QUERY)):
            cid=rng.choice(all_ids)
            if cid not in gold and cid not in negatives: negatives.append(cid)
            if len(negatives)>=RANDOM_NEGATIVES_PER_QUERY: break
        if RANDOM_NEGATIVES_PER_QUERY == 0: negatives=[]
        if len(negatives)<RANDOM_NEGATIVES_PER_QUERY:
            negatives += [c for c in all_ids if c not in gold and c not in negatives][:RANDOM_NEGATIVES_PER_QUERY-len(negatives)]
        cids.extend(negatives)
    cids=list(dict.fromkeys(cids))
    mask=torch.tensor([[cid in queries[qid]['gold'] for cid in cids] for qid in query_ids],dtype=torch.bool)
    if not mask.any(1).all(): raise RuntimeError('A training query has no positive in its candidate set.')
    if mask.all(1).any():
        raise ValueError('A contrastive query has no negative. Increase negatives/batch size or audit all-positive gold sets.')
    return list(query_ids),cids,mask


## Section 5 · Adaptation, dual encoder, InfoNCE and knowledge distillation

Queries and candidates share the same backbone, pooling and projection. Each forward uses one unpadded window. Causal models retain causal attention; encoder retrieval is bidirectional. Content-token means exclude prompt and EOS positions; case pooling is mean or trainable attention, followed by a learned projection and L2 normalization.

- **LORMA:** exact output-side `W'=(I+(alpha/r)BA)W0`; frozen bias is unchanged. No additive LoRA is substituted.
- **QLoRA:** two NF4 codes per byte plus FP32 block scales, frozen base and trainable additive LoRA. Custom autograd regenerates dense weights in backward instead of retaining all dequantized weights. This is a portable NF4+LoRA implementation, **without double quantization, paged optimizers or fused kernels**; embeddings/norms/SSM dynamics remain unquantized. It may run slower than FP16-base PEFT.
- **Adapters:** one post-block residual bottleneck, a Pfeiffer-style placement. Applying this after Mamba blocks is an explicit hybrid extension.
- **Prompt/prefix:** causal soft input embeddings; encoder learned per-layer K/V prefixes in post-RoPE attention space. Prefix positions do not consume document RoPE positions.
- **IA3:** attention K/V outputs and FFN intermediate inputs; hybrid SSM dynamics are left unchanged.
- **KD:** train a full-depth full-FT retrieval teacher, cache only training query/candidate embeddings, and prune odd zero-based layers to create a half-depth student. Transfer the trained projection/pooler. For a 20-layer MMMA hybrid the student is five Mamba plus five attention layers; this changes their ratio and is logged. Encoder KD halves only the active encoder. Teacher and student see the exact same sampled candidate ID order. The loss is `(1-alpha)*InfoNCE + alpha*T²*KL(teacher || student)`; cached embeddings produce soft similarity targets, not external labels.

InfoNCE marks **all known positives in the sampled candidate matrix** to avoid treating another query's shared gold as a negative. Candidate IDs are deduplicated before encoding. Gradient accumulation does not enlarge that matrix. Two-pass chunk-feature gradient replay preserves RNG states and limits live backbone graphs to one window; its gradients are compared against ordinary backprop in development tests. The retrieval wrapper optionally checkpoints individual blocks, without invoking the original mask-dropping checkpoint paths.

The hybrid's SSM states are fixed-size per layer, but its attention is not constant-memory across arbitrary lengths. PCR uses parallel SSD evaluation inside bounded windows; token-by-token `step()` decoding is unnecessary for embedding extraction.


In [45]:
# Section 5 — real adaptation modules, shared bi-encoder, InfoNCE, KD and gradient replay.
class LowRankLinear(nn.Module):
    def __init__(self, base, method='lorma'):
        super().__init__(); self.base=base; self.method=method
        self.in_features=base.in_features; self.out_features=base.out_features
        self.base.requires_grad_(False); self.scale=LORA_ALPHA/RANK
        width=self.out_features if method=='lorma' else self.in_features
        self.A=nn.Parameter(torch.empty(RANK,width)); self.B=nn.Parameter(torch.zeros(self.out_features,RANK))
        nn.init.kaiming_uniform_(self.A,a=math.sqrt(5))
    def forward(self,x):
        if self.method=='lorma':
            y=F.linear(x,self.base.weight,None)
            y=y+self.scale*F.linear(F.linear(y,self.A),self.B)
            return y if self.base.bias is None else y+self.base.bias
        return self.base(x)+self.scale*F.linear(F.linear(x,self.A),self.B)

# Standard NF4 codebook; pure PyTorch packing, no bitsandbytes or pretrained weights.
NF4_VALUES=[-1.,-.6961928009986877,-.5250730514526367,-.39491748809814453,
 -.28444138169288635,-.18477343022823334,-.09105003625154495,0.,.07958029955625534,
 .16093020141124725,.24611230194568634,.33791524171829224,.44070982933044434,
 .5626170039176941,.7229568362236023,1.]

def pack_nf4(weight,blocksize=64):
    flat=weight.detach().float().cpu().flatten(); pad=(-flat.numel())%blocksize
    blocks=F.pad(flat,(0,pad)).reshape(-1,blocksize)
    scales=blocks.abs().amax(1).clamp_min(1e-12)
    codebook=torch.tensor(NF4_VALUES)
    boundaries=(codebook[:-1]+codebook[1:])/2
    codes=torch.bucketize((blocks/scales[:,None]).contiguous(),boundaries).to(torch.uint8).flatten()
    packed=(codes[0::2] | (codes[1::2]<<4)).contiguous()
    return packed,scales

def unpack_nf4(packed,scales,shape,blocksize,dtype):
    codes=torch.stack((packed & 15,packed >> 4),dim=-1).flatten().long()
    table=torch.tensor(NF4_VALUES,device=packed.device,dtype=torch.float32)
    values=table[codes].reshape(-1,blocksize)*scales[:,None]
    return values.flatten()[:math.prod(shape)].reshape(shape).to(dtype)

class NF4FrozenFunction(torch.autograd.Function):
    # Save packed codes only: regenerate transient dense weights during backward.
    @staticmethod
    @torch.amp.custom_fwd(device_type='cuda')
    def forward(ctx,x,packed,scales,bias,shape,blocksize):
        ctx.save_for_backward(packed,scales); ctx.shape=shape;ctx.blocksize=blocksize;ctx.input_dtype=x.dtype
        weight=unpack_nf4(packed,scales,shape,blocksize,x.dtype)
        return F.linear(x,weight,bias)
    @staticmethod
    @torch.amp.custom_bwd(device_type='cuda')
    def backward(ctx,dy):
        packed,scales=ctx.saved_tensors
        weight=unpack_nf4(packed,scales,ctx.shape,ctx.blocksize,dy.dtype)
        dx=torch.matmul(dy,weight).to(ctx.input_dtype) if ctx.needs_input_grad[0] else None
        return dx,None,None,None,None,None

class NF4LoRALinear(nn.Module):
    def __init__(self,base):
        super().__init__();self.in_features=base.in_features;self.out_features=base.out_features
        self.shape=tuple(base.weight.shape);self.blocksize=NF4_BLOCK_SIZE
        packed,scales=pack_nf4(base.weight,self.blocksize)
        self.register_buffer('packed',packed);self.register_buffer('scales',scales)
        self.register_buffer('bias',None if base.bias is None else base.bias.detach().clone())
        self.A=nn.Parameter(torch.empty(RANK,self.in_features));self.B=nn.Parameter(torch.zeros(self.out_features,RANK))
        nn.init.kaiming_uniform_(self.A,a=math.sqrt(5));self.scale=LORA_ALPHA/RANK
    def forward(self,x):
        y=NF4FrozenFunction.apply(x,self.packed,self.scales,self.bias,self.shape,self.blocksize)
        return y+self.scale*F.linear(F.linear(x,self.A),self.B)

class IA3Linear(nn.Module):
    def __init__(self,base,on_input=False):
        super().__init__();self.base=base;self.on_input=on_input;self.base.requires_grad_(False)
        self.in_features=base.in_features;self.out_features=base.out_features
        self.scale=nn.Parameter(torch.ones(base.in_features if on_input else base.out_features))
    def forward(self,x):
        return self.base(x*self.scale) if self.on_input else self.base(x)*self.scale

class ResidualAdapter(nn.Module):
    def __init__(self,dim):
        super().__init__();self.norm=nn.LayerNorm(dim);self.down=nn.Linear(dim,ADAPTER_DIM)
        self.up=nn.Linear(ADAPTER_DIM,dim);self.dropout=nn.Dropout(ADAPTER_DROPOUT)
        nn.init.zeros_(self.up.weight);nn.init.zeros_(self.up.bias)
    def forward(self,x): return x+self.up(self.dropout(F.gelu(self.down(self.norm(x)))))

class AdapterBlock(nn.Module):
    """One post-block bottleneck (Pfeiffer-style placement); hybrid is an explicit extension."""
    def __init__(self,base,dim):
        super().__init__();self.base=base;self.adapter=ResidualAdapter(dim)
        self.kind=getattr(base,'kind','A')
    def forward(self,*args,**kwargs):
        result=self.base(*args,**kwargs)
        if isinstance(result,tuple): return (self.adapter(result[0]),*result[1:])
        return self.adapter(result)

class EncoderPrefixAttention(nn.Module):
    """Learned per-layer post-RoPE K/V prefix, bidirectional encoder self-attention only."""
    def __init__(self,base):
        super().__init__();self.base=base;self.base.requires_grad_(False)
        self.key=nn.Parameter(torch.randn(1,base.nh,KV_PREFIX_TOKENS,base.hd)*0.02)
        self.value=nn.Parameter(torch.randn(1,base.nh,KV_PREFIX_TOKENS,base.hd)*0.02)
    def forward(self,x,ctx=None,cos=None,sin=None,mask=None,causal=False,cache=None,kv=None):
        if ctx is not None or causal or cache is not None or kv is not None:
            raise ValueError('Prefix wrapper is encoder self-attention only.')
        b,t,_=x.shape;a=self.base
        q=a._split(a.q(x));k,v=a.project_kv(x)
        if a.rope: q,k=apply_rope(q,cos,sin),apply_rope(k,cos,sin)
        k=torch.cat((self.key.to(k.dtype).expand(b,-1,-1,-1),k),dim=2)
        v=torch.cat((self.value.to(v.dtype).expand(b,-1,-1,-1),v),dim=2)
        if mask is not None:
            prefix=torch.ones((*mask.shape[:-1],KV_PREFIX_TOKENS),device=mask.device,dtype=mask.dtype)
            if mask.dtype!=torch.bool: prefix.zero_()
            mask=torch.cat((prefix,mask),-1)
        y=F.scaled_dot_product_attention(q,k,v,attn_mask=mask,is_causal=False)
        return a.o(y.transpose(1,2).reshape(b,t,-1)),None

def replace_linears(module,factory):
    count=0
    for name,child in list(module.named_children()):
        if isinstance(child,nn.Linear): setattr(module,name,factory(child));count+=1
        else: count+=replace_linears(child,factory)
    return count

def set_layers(base,layers):
    if BACKBONE=='encdec': base.enc_blocks=nn.ModuleList(layers)
    else: base.blocks=nn.ModuleList(layers)

def adapt_backbone(base,preset):
    method=preset.method; base.requires_grad_(method in {'full','kd'})
    layers=retrieval_layers(base); count=0
    if method=='partial':
        for layer in list(layers)[-preset.train_top:]: layer.requires_grad_(True)
        if TRAIN_FINAL_NORM_IN_PARTIAL: retrieval_final_norm(base).requires_grad_(True)
        if TRAIN_EMBEDDINGS_IN_PARTIAL: embedding_layer(base).requires_grad_(True)
    elif method in {'lorma','lora','qlora'}:
        factory=(lambda layer:NF4LoRALinear(layer)) if method=='qlora' else (lambda layer:LowRankLinear(layer,method))
        for layer in layers: count+=replace_linears(layer,factory)
    elif method=='adapter':
        set_layers(base,[AdapterBlock(layer,hidden_size(base.c)) for layer in layers]);count=len(layers)
    elif method=='ia3':
        for layer in layers:
            if hasattr(layer,'attn'):
                layer.attn.k=IA3Linear(layer.attn.k);layer.attn.v=IA3Linear(layer.attn.v);count+=2
            if hasattr(layer,'mlp'):
                layer.mlp.down=IA3Linear(layer.mlp.down,on_input=True);count+=1
        if not count: raise ValueError('No attention/FFN modules are available for IA3.')
    elif method=='prefix':
        if BACKBONE!='encdec': raise ValueError('Use encoder prefix or causal prompt preset.')
        for layer in layers: layer.attn=EncoderPrefixAttention(layer.attn);count+=1
    elif method not in {'full','kd','partial','prompt'}: raise ValueError(f'Unknown method {method}')
    return count

class RetrievalModel(nn.Module):
    def __init__(self,base,preset):
        super().__init__();self.backbone=base;self.preset=preset
        self.adapted_modules=adapt_backbone(base,preset);dim=hidden_size(base.c)
        self.proj=nn.Linear(dim,EMBED_DIM,bias=False);self.head_dropout=nn.Dropout(HEAD_DROPOUT)
        self.pool=nn.Linear(dim,1,bias=False) if CHUNK_POOL=='attention' else None
        self.prompt=nn.Parameter(torch.randn(SOFT_PROMPT_TOKENS,dim)*0.02) if preset.method=='prompt' else None
        self.checkpoint_blocks=RETRIEVAL_GRAD_CKPT
    def chunk_features(self,tokens):
        # One unpadded window: no padding can corrupt attention or the SSM recurrence.
        device=next(self.parameters()).device;ids=torch.tensor(tokens,device=device,dtype=torch.long)[None]
        count=ids.shape[1];base=self.backbone
        if APPEND_EOS_PER_CHUNK:
            ids=torch.cat((ids,torch.full((1,1),TOKENIZER.eos_id,device=device,dtype=torch.long)),1)
        x=embedding_layer(base)(ids);offset=0
        if self.prompt is not None:
            x=torch.cat((self.prompt[None].to(x.dtype),x),1);offset=len(self.prompt)
        if x.size(1)>base.c.n_ctx: raise ValueError('Window exceeds checkpoint context.')
        cos,sin=base.cos[:x.size(1)][None,None],base.sin[:x.size(1)][None,None]
        for block in retrieval_layers(base):
            def run(value,block=block):
                if BACKBONE=='encdec': return block(value,cos,sin,None)
                if BACKBONE=='mamba': return block(value,cos,sin,None,None,False)[0]
                return block(value,cos,sin,None,None)[0]
            if self.training and torch.is_grad_enabled() and self.checkpoint_blocks:
                x=torch.utils.checkpoint.checkpoint(run,x,use_reentrant=False,preserve_rng_state=True)
            else: x=run(x)
        x=retrieval_final_norm(base)(x)
        # Prompt and EOS are context, not content pooling tokens.
        return x[:,offset:offset+count].float().mean(1).squeeze(0)
    def aggregate(self,features):
        f=features.float()
        if self.pool is None: pooled=f.mean(0)
        else:
            scores=self.pool(f).float().squeeze(-1)
            pooled=(f*F.softmax(scores,dim=0)[:,None]).sum(0)
        z=self.proj(self.head_dropout(pooled)).float()
        if not torch.isfinite(z).all(): raise FloatingPointError('Non-finite retrieval embedding.')
        return F.normalize(z,dim=-1,eps=1e-8)
    def encode_document(self,record,store):
        return self.aggregate(torch.stack([self.chunk_features(c) for c in store[record]]))

@torch.no_grad()
def embed_records(model,records,store,progress=''):
    model.eval();out=[]
    for i,record in enumerate(records):
        with amp_context(): out.append(model.encode_document(record,store).float().cpu())
        if progress and (i+1)%100==0: print(progress,i+1,'/',len(records),flush=True)
    if not out: return torch.empty(0,EMBED_DIM)
    result=torch.stack(out)
    if not torch.isfinite(result).all(): raise FloatingPointError('Index has non-finite values.')
    return result

def contrastive_kd_loss(q,c,positive,teacher_scores=None):
    scores=(q.float()@c.float().T)/INFO_NCE_TEMPERATURE
    positive=positive.to(scores.device)
    nce=(torch.logsumexp(scores,1)-torch.logsumexp(scores.masked_fill(~positive,-torch.inf),1)).mean()
    if teacher_scores is None: return nce
    temp=KD_TEMPERATURE
    teacher_probs=F.softmax(teacher_scores.to(scores.device).float()/temp,dim=-1)
    kd=F.kl_div(F.log_softmax(scores/temp,dim=-1),teacher_probs,reduction='batchmean')*temp*temp
    return (1-KD_ALPHA)*nce+KD_ALPHA*kd

def rng_snapshot():
    return (torch.get_rng_state(),torch.cuda.get_rng_state(DEVICE) if DEVICE.type=='cuda' else None)

def backward_batch(model,query_records,candidate_records,positive,store,scaler,divisor=1,teacher_scores=None):
    model.train();records=query_records+candidate_records;nq=len(query_records)
    if not USE_GRAD_CACHE:
        with amp_context():
            embeddings=torch.stack([model.encode_document(r,store) for r in records])
            loss=contrastive_kd_loss(embeddings[:nq],embeddings[nq:],positive,teacher_scores)
        scaler.scale(loss/divisor).backward();return float(loss.detach())
    # Pass 1: features without graphs. Only small chunk-feature leaves and head graphs are retained.
    replay=[];embeddings=[]
    for record in records:
        values=[];states=[];chunks=store[record]
        for chunk in chunks:
            states.append(rng_snapshot())
            with torch.no_grad(),amp_context(): values.append(model.chunk_features(chunk))
        leaf=torch.stack(values).detach().requires_grad_(True)
        with amp_context(): embeddings.append(model.aggregate(leaf))
        replay.append((chunks,states,leaf))
    with amp_context():
        z=torch.stack(embeddings);loss=contrastive_kd_loss(z[:nq],z[nq:],positive,teacher_scores)
    if not torch.isfinite(loss): raise FloatingPointError('Non-finite contrastive/KD loss.')
    scaler.scale(loss/divisor).backward()  # Head gradients + SCALED chunk-feature gradients.
    # Pass 2: replay each chunk with identical dropout RNG; preserve the advanced global RNG.
    devices=[DEVICE.index or 0] if DEVICE.type=='cuda' else []
    for chunks,states,leaf in replay:
        for i,(chunk,state) in enumerate(zip(chunks,states)):
            with torch.random.fork_rng(devices=devices):
                torch.set_rng_state(state[0])
                if DEVICE.type=='cuda': torch.cuda.set_rng_state(state[1],DEVICE)
                with amp_context(): feature=model.chunk_features(chunk)
                if feature.requires_grad:
                    torch.autograd.backward(feature,leaf.grad[i])  # Do not scale twice.
    return float(loss.detach())

def model_counts(model):
    return {'total_parameters':sum(p.numel() for p in model.parameters()),
            'trainable_parameters':sum(p.numel() for p in model.parameters() if p.requires_grad),
            'buffer_bytes':sum(b.numel()*b.element_size() for b in model.buffers())}

def make_student(teacher,preset):
    base=copy.deepcopy(teacher.backbone).cpu();old=list(retrieval_layers(base))
    keep=list(range(1,len(old),2))
    if not keep: raise ValueError('Cannot halve a one-layer backbone.')
    set_layers(base,[old[i] for i in keep])
    if BACKBONE=='encdec': base.c.n_enc=len(keep)
    elif BACKBONE=='decoder': base.c.n_layer=len(keep)
    else: base.c.pattern=''.join(base.c.pattern[i] for i in keep)
    student=RetrievalModel(base,preset)
    student.proj.load_state_dict(teacher.proj.state_dict())
    if student.pool is not None: student.pool.load_state_dict(teacher.pool.state_dict())
    if model_counts(student)['total_parameters']>=model_counts(teacher)['total_parameters']:
        raise ValueError('KD student is not smaller than the active full-depth teacher.')
    print('KD retained zero-based layer indices:',keep,'| new config:',asdict(base.c))
    return student,keep


## Section 6 · Full-pool ranking and Micro-F1@7

Embeddings and cosine scores are FP32. Candidate score blocks bound ranking memory; all candidates are considered. Ties use lexicographic candidate IDs. Only predictions are limited to seven—**never truncate or pad gold sets**.

Across queries: `TP = sum(|predicted_7 ∩ gold|)`, `P = TP/(7N)`, `R = TP/sum(|gold|)`, `F1 = 2TP/(7N + sum(|gold|))`. Every dev prediction must contain seven unique candidate IDs from the dev pool.


In [46]:
# Section 6 — exact top-k retrieval and official micro counts.
def rank_embeddings(query_ids,qemb,candidate_ids,cemb,k=7):
    if len(candidate_ids)<k: raise ValueError(f'Candidate pool has fewer than {k} cases.')
    if len(set(query_ids))!=len(query_ids) or len(set(candidate_ids))!=len(candidate_ids):
        raise ValueError('Duplicate indexing IDs.')
    if qemb.shape!=(len(query_ids),EMBED_DIM) or cemb.shape!=(len(candidate_ids),EMBED_DIM):
        raise ValueError('Index dimensions do not match IDs.')
    if not torch.isfinite(qemb).all() or not torch.isfinite(cemb).all():
        raise FloatingPointError('Non-finite index embeddings.')
    order=sorted(range(len(candidate_ids)),key=lambda i:candidate_ids[i])
    ids=[candidate_ids[i] for i in order];C=cemb[order].float().cpu().numpy()
    Q=qemb.float().cpu().numpy();pred={}
    for first in range(0,len(query_ids),EVAL_QUERY_BLOCK_SIZE):
        q=Q[first:first+EVAL_QUERY_BLOCK_SIZE]
        best_s=np.empty((len(q),0),dtype=np.float32);best_i=np.empty((len(q),0),dtype=np.int64)
        for start in range(0,len(ids),SCORE_CANDIDATE_BLOCK_SIZE):
            stop=min(start+SCORE_CANDIDATE_BLOCK_SIZE,len(ids));scores=q@C[start:stop].T
            indices=np.broadcast_to(np.arange(start,stop),(len(q),stop-start))
            scores=np.concatenate((best_s,scores),axis=1);indices=np.concatenate((best_i,indices),axis=1)
            # Exact deterministic top-k: descending cosine, then lexicographic candidate ID.
            winners=np.lexsort((indices,-scores),axis=1)[:,:k]
            best_s=np.take_along_axis(scores,winners,axis=1);best_i=np.take_along_axis(indices,winners,axis=1)
        for row,qid in enumerate(query_ids[first:first+len(q)]): pred[qid]=[ids[i] for i in best_i[row]]
    return pred

def validate_predictions(pred,queries,candidates):
    if not isinstance(pred,dict) or set(pred)!=set(queries):
        raise ValueError('Prediction keys must exactly match every query ID, with no extras.')
    pool=set(candidates)
    for qid,ranked in pred.items():
        if not isinstance(qid,str) or not isinstance(ranked,list) or len(ranked)!=7:
            raise ValueError(f'{qid}: prediction must be a list of EXACTLY 7 IDs.')
        if not all(isinstance(cid,str) for cid in ranked) or len(set(ranked))!=7:
            raise ValueError(f'{qid}: candidate IDs must be unique strings.')
        if not set(ranked)<=pool: raise ValueError(f'{qid}: candidate is outside this split\'s pool.')
    return True

def micro_metrics(pred,queries,candidates):
    validate_predictions(pred,queries,candidates)
    tp=sum(len(set(pred[qid]) & q['gold']) for qid,q in queries.items())
    predicted=7*len(queries);gold_total=sum(len(q['gold']) for q in queries.values())
    return {'micro_f1_at_7':2*tp/(predicted+gold_total) if predicted+gold_total else 0.,
            'micro_precision_at_7':tp/predicted if predicted else 0.,
            'micro_recall_at_7':tp/gold_total if gold_total else 0.,
            'true_positives':tp,'predicted_total':predicted,'gold_total':gold_total,'queries':len(queries)}

def evaluate(model,queries,candidates,store,label='dev'):
    ids=list(queries);cids=sorted(candidates)
    q=embed_records(model,[queries[i] for i in ids],store,label+' queries')
    c=embed_records(model,[candidates[i] for i in cids],store,label+' candidates')
    pred=rank_embeddings(ids,q,cids,c)
    metrics=micro_metrics(pred,queries,candidates)
    print(label,metrics,flush=True)
    return metrics

def dense_hard_negatives(model,data,store):
    queries=data['train_queries'];candidates=data['train_candidates'];qids=list(queries);cids=sorted(candidates)
    q=embed_records(model,[queries[i] for i in qids],store,'Mining train queries')
    c=embed_records(model,[candidates[i] for i in cids],store,'Mining train candidates')
    k=min(len(cids),HARD_MINING_TOPK+max(len(v['gold']) for v in queries.values()))
    ranked=rank_embeddings(qids,q,cids,c,k=k)
    return {i:[cid for cid in ranked[i] if cid not in queries[i]['gold']][:HARD_MINING_TOPK] for i in qids}


## Section 7 · Benchmark, budget-aware training and dev evaluation

The benchmark runs real forward/backward/optimizer work with representative sampled candidate sets, then restores model weights. It reports seconds/query and seconds per scored query–candidate pair (shared encoding; not independent full document passes). Inference benchmarks use long retained windows. `TRAIN_QUERIES_PER_EPOCH` is auto-fit from the remaining budget after reserving dev, dense mining, teacher-cache, test indexing and checkpoint time. Initial CPU parsing/tokenization is also included in the wall-clock budget.

Teacher and student occupy the GPU sequentially. Teacher cached embeddings reside on CPU/disk. Each student epoch evaluates against the complete dev candidate pool; the best Micro-F1@7 checkpoint is restored before test ranking. With `MAX_DEV_QUERIES=None`, all dev queries are evaluated. If you explicitly cap dev queries, the result is labeled a subset score.

Budget fitting does not truncate a candidate pool or return fewer than seven results. If mandatory work alone exceeds the budget, increase `TOTAL_GPU_MINUTES` or reduce `MAX_CHUNKS` and rerun. The controller stops training early to preserve inference time, but reports an actual overrun honestly rather than asserting a hard deadline was met. Checkpoints are for reproducible retrieval/reloading, not optimizer-state resumption.


In [47]:
# Section 7 — measured throughput, optimizer updates, teacher cache, budget-aware fitting.
class BudgetError(RuntimeError): pass

def optimizer_for(model,preset):
    head=[];body=[]
    for name,p in model.named_parameters():
        if p.requires_grad:
            (head if name.startswith(('proj.','pool.')) else body).append(p)
    groups=[]
    if body: groups.append({'params':body,'lr':preset.lr_backbone,'base_lr':preset.lr_backbone})
    if head: groups.append({'params':head,'lr':preset.lr_head,'base_lr':preset.lr_head})
    return torch.optim.AdamW(groups,weight_decay=WEIGHT_DECAY,foreach=False)

def step_optimizer(model,opt,scaler):
    scaler.unscale_(opt)
    parameters=[p for p in model.parameters() if p.requires_grad and p.grad is not None]
    norm=torch.nn.utils.clip_grad_norm_(parameters,CLIP_NORM)
    old_scale=scaler.get_scale()
    if DEVICE.type!='cuda' and not torch.isfinite(norm): raise FloatingPointError('Non-finite gradients.')
    scaler.step(opt);scaler.update();opt.zero_grad(set_to_none=True)
    return scaler.get_scale()>=old_scale

def benchmark_training(model,data,store,hard):
    snapshot={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
    opt=optimizer_for(model,model.preset);scaler=make_grad_scaler();rng=random.Random(SEED+13)
    query_ids=list(data['train_queries']);secs=[];pairs=[];queries=[]
    try:
        for i in range(BENCHMARK_WARMUP_STEPS+BENCHMARK_STEPS):
            selected=rng.sample(query_ids,min(CONTRASTIVE_BATCH_QUERIES,len(query_ids)))
            qids,cids,positive=sample_batch(selected,data['train_queries'],data['train_candidates'],hard,rng)
            # Include representative sampled-set KD arithmetic for KD presets.
            targets=torch.zeros(len(qids),len(cids)) if model.preset.method=='kd' else None
            sync_device();start=time.monotonic()
            loss=backward_batch(model,[data['train_queries'][q] for q in qids],
                [data['train_candidates'][c] for c in cids],positive,store,scaler,teacher_scores=targets)
            success=step_optimizer(model,opt,scaler);sync_device();elapsed=time.monotonic()-start
            if not success: print('Benchmark FP16 overflow: scaler reduced; retrying at lower scale.')
            if i>=BENCHMARK_WARMUP_STEPS:
                secs.append(elapsed);pairs.append(len(qids)*len(cids));queries.append(len(qids))
        result={'seconds_per_query':max(s/q for s,q in zip(secs,queries)),
                'seconds_per_scored_pair':sum(secs)/sum(pairs),
                'median_seconds_per_microbatch':float(np.median(secs)),
                'pair_definition':'query-by-candidate scores in a sampled contrastive matrix; shared encoding',
                'sampled_queries':sum(queries),'optimizer_step_included':True}
    finally:
        model.load_state_dict(snapshot,strict=True);model.zero_grad(set_to_none=True)
        del snapshot,opt,scaler;gc.collect()
        if DEVICE.type=='cuda': torch.cuda.empty_cache()
    print('Training benchmark:',result,flush=True)
    return result

@torch.no_grad()
def benchmark_inference(model,data,store):
    # Include the longest retained windows, not just short first records.
    records=[r for rs in data.values() for r in rs.values()]
    records.sort(key=lambda r:sum(map(len,store[r])),reverse=True)
    chosen=records[:min(INFERENCE_BENCHMARK_DOCS,len(records))]
    model.eval()
    with amp_context(): model.encode_document(chosen[0],store)
    sync_device();start=time.monotonic();chunks=0
    for record in chosen:
        with amp_context(): model.encode_document(record,store)
        chunks+=len(store[record])
    sync_device()
    sec=(time.monotonic()-start)/max(1,chunks)
    print('Inference benchmark seconds/chunk:',round(sec,6),flush=True)
    return max(sec,1e-6)

def count_chunks(records,store): return sum(len(store[r]) for r in records.values())

def index_seconds(queries,candidates,store,seconds_per_chunk):
    chunks=count_chunks(queries,store)+count_chunks(candidates,store)
    # CPU ranking estimate measured on an actual score block, then extrapolated.
    nq=min(EVAL_QUERY_BLOCK_SIZE,len(queries));nc=min(SCORE_CANDIDATE_BLOCK_SIZE,len(candidates))
    rng=np.random.default_rng(SEED)
    q=rng.normal(size=(nq,EMBED_DIM)).astype(np.float32);c=rng.normal(size=(nc,EMBED_DIM)).astype(np.float32)
    start=time.monotonic();s=q@c.T;np.argsort(-s,axis=1,kind='stable')
    ranking=(time.monotonic()-start)*math.ceil(len(queries)/max(1,nq))*math.ceil(len(candidates)/max(1,nc))
    return BENCHMARK_MARGIN*(chunks*seconds_per_chunk+ranking)

def reserve_inference(data,dev,store,seconds_per_chunk,epochs,kd_cache=False):
    dev_sec=index_seconds(dev,data['dev_candidates'],store,seconds_per_chunk)
    test_sec=index_seconds(data['test_queries'],data['test_candidates'],store,seconds_per_chunk)
    mining_sec=index_seconds(data['train_queries'],data['train_candidates'],store,seconds_per_chunk)
    refreshes=sum(1 for e in range(1,epochs) if HARD_REFRESH_EPOCHS>0 and e%HARD_REFRESH_EPOCHS==0)
    return {'dev_per_epoch':dev_sec,'test':test_sec,'mining_per_refresh':mining_sec,
            'refreshes':refreshes,'teacher_cache':mining_sec if kd_cache else 0.,
            'total':epochs*dev_sec+test_sec+refreshes*mining_sec+(mining_sec if kd_cache else 0.)+CHECKPOINT_RESERVE_SECONDS}

def save_retrieval(model,path,meta,metrics=None,extra=None):
    payload={'format':'cs781_pcr_v1','backbone':BACKBONE,'cfg':asdict(model.backbone.c),
             'preset':asdict(model.preset),'model':{k:v.detach().cpu() for k,v in model.state_dict().items()},
             'assets':meta,'embedding_dim':EMBED_DIM,'chunk_pool':CHUNK_POOL,
             'token_signature':TOKEN_STORE.signature,'data_fingerprints':DATA_FINGERPRINTS,
             'metrics':metrics,'extra':extra or {}}
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True);tmp=path.with_suffix('.tmp')
    torch.save(payload,tmp);os.replace(tmp,path)

def restore_retrieval(model,path):
    ck=torch.load(path,map_location='cpu',weights_only=False)
    model.load_state_dict(ck['model'],strict=True)
    return ck

def fit_stage(model,data,dev,store,hard,preset,train_seconds,benchmark,meta,
              cache=None,epochs=EPOCHS,checkpoint_path=None,global_deadline=None,reserve=None):
    per_query=benchmark['seconds_per_query']*BENCHMARK_MARGIN
    cap=min(len(data['train_queries']),int(train_seconds/max(1,epochs)/per_query))
    if TRAIN_QUERIES_PER_EPOCH is not None: cap=min(cap,TRAIN_QUERIES_PER_EPOCH)
    if not AUTO_FIT_BUDGET:
        cap=min(len(data['train_queries']),TRAIN_QUERIES_PER_EPOCH or len(data['train_queries']))
        if cap*epochs*per_query>train_seconds: raise BudgetError('Requested fixed training count exceeds the measured budget.')
    if cap<1: raise BudgetError('Budget cannot fit one training query/epoch plus mandatory indexing.')
    print('AUTO-FIT TRAIN_QUERIES_PER_EPOCH =',cap,'| epochs =',epochs,flush=True)
    opt=optimizer_for(model,preset);scaler=make_grad_scaler();history=[];best=-math.inf
    rng=random.Random(SEED+71);updates=0;successful=0;queries_seen=0
    expected_updates=max(1,epochs*math.ceil(math.ceil(cap/CONTRASTIVE_BATCH_QUERIES)/GRAD_ACCUM))
    warmup=max(1,int(expected_updates*WARMUP_FRACTION))
    for epoch in range(epochs):
        epoch_limit=train_seconds/epochs
        if global_deadline is not None and reserve:
            remaining_eval=(epochs-epoch)*reserve['dev_per_epoch']+reserve['test']+CHECKPOINT_RESERVE_SECONDS
            remaining_mining=sum(1 for e in range(max(1,epoch),epochs) if HARD_REFRESH_EPOCHS>0 and e%HARD_REFRESH_EPOCHS==0)*reserve['mining_per_refresh']
            epoch_limit=min(epoch_limit,max(0.,global_deadline-time.monotonic()-remaining_eval-remaining_mining))
        if epoch_limit<per_query: print('Stopping training to preserve the submission budget.');break
        if epoch and HARD_REFRESH_EPOCHS>0 and epoch%HARD_REFRESH_EPOCHS==0:
            hard=dense_hard_negatives(model,data,store)
        selected=list(data['train_queries']);rng.shuffle(selected);selected=selected[:cap]
        batches=[selected[i:i+CONTRASTIVE_BATCH_QUERIES] for i in range(0,len(selected),CONTRASTIVE_BATCH_QUERIES)]
        start=time.monotonic();deadline=start+epoch_limit;losses=[];epoch_queries=0
        for first in range(0,len(batches),GRAD_ACCUM):
            group=batches[first:first+GRAD_ACCUM]
            estimated=sum(map(len,group))*per_query
            if time.monotonic()+estimated>deadline and epoch_queries: break
            ratio=min(1.,(updates+1)/warmup)
            ratio*=max(.1,.5*(1+math.cos(math.pi*max(0,updates-warmup)/max(1,expected_updates-warmup))))
            for g in opt.param_groups:g['lr']=g['base_lr']*ratio
            opt.zero_grad(set_to_none=True)
            for qids0 in group:
                qids,cids,positive=sample_batch(qids0,data['train_queries'],data['train_candidates'],hard,rng)
                scores=None if cache is None else cache.scores(qids,cids)
                loss=backward_batch(model,[data['train_queries'][q] for q in qids],
                    [data['train_candidates'][c] for c in cids],positive,store,scaler,len(group),scores)
                losses.append(loss);epoch_queries+=len(qids)
            success=step_optimizer(model,opt,scaler);updates+=1;successful+=int(success)
            if updates%LOG_EVERY_UPDATES==0:
                print(f'epoch {epoch+1} | update {updates} | queries {epoch_queries} | loss {np.mean(losses[-10:]):.4f}',flush=True)
        queries_seen+=epoch_queries
        row={'epoch':epoch+1,'queries':epoch_queries,'loss':float(np.mean(losses)) if losses else None,
             'training_seconds':time.monotonic()-start,'optimizer_updates':updates,'successful_updates':successful}
        if dev is not None:
            row['dev']=evaluate(model,dev,data['dev_candidates'],store)
            score=row['dev'][PRIMARY_METRIC]
        else: score=-(row['loss'] if row['loss'] is not None else math.inf)
        if checkpoint_path and score>best:
            best=score;save_retrieval(model,checkpoint_path,meta,row.get('dev'),{'epoch':epoch+1})
        history.append(row);print('Epoch completed:',row,flush=True)
    del opt;gc.collect()
    if successful==0: raise RuntimeError('No successful optimizer updates; refusing to label an untrained run as trained.')
    if checkpoint_path:
        if not Path(checkpoint_path).is_file(): raise RuntimeError('No best checkpoint was saved.')
        restore_retrieval(model,checkpoint_path)
    return {'auto_fit_queries_per_epoch':cap,'queries_seen':queries_seen,'history':history,
            'optimizer_updates':updates,'successful_updates':successful}

class TeacherCache:
    def __init__(self,qids,q,cids,c,meta):
        self.qmap={x:i for i,x in enumerate(qids)};self.cmap={x:i for i,x in enumerate(cids)}
        self.q=q.float().cpu();self.c=c.float().cpu();self.meta=meta
    def scores(self,qids,cids):
        # Exact alignment of the sampled candidate ID order for both student and teacher.
        return (self.q[[self.qmap[i] for i in qids]]@self.c[[self.cmap[i] for i in cids]].T)/INFO_NCE_TEMPERATURE

def cache_teacher(teacher,data,store,meta,teacher_checkpoint):
    qids=list(data['train_queries']);cids=sorted(data['train_candidates'])
    q=embed_records(teacher,[data['train_queries'][i] for i in qids],store,'Teacher queries')
    c=embed_records(teacher,[data['train_candidates'][i] for i in cids],store,'Teacher candidates')
    cache_meta={'teacher_checkpoint_sha256':sha256_file(teacher_checkpoint),'token_signature':store.signature,
                'tokenizer_sha256':meta['tokenizer_sha256'],'split':'train','score_temperature':INFO_NCE_TEMPERATURE,
                'data_fingerprints':DATA_FINGERPRINTS,'qids':qids,'cids':cids,
                'signal':'sampled_candidate_similarity_distribution'}
    TEACHER_CACHE_DIR.mkdir(parents=True,exist_ok=True)
    torch.save({'q':q,'c':c,'meta':cache_meta},TEACHER_CACHE_DIR/'embeddings.pt')
    return TeacherCache(qids,q,cids,c,cache_meta)


## Section 8 · Test indexing and validated submission

The final model independently embeds all test queries and every test candidate, ranks by cosine similarity and writes `prediction.json` as a JSON object. Validation checks exact query-key coverage, string IDs, seven distinct candidates per query, and membership in the test pool. The ZIP is reopened and checked to contain **only `prediction.json` at its root**. The run copy and a convenient `/kaggle/working/submission_<EXP>.zip` copy are retained.


In [48]:
# Section 8 — full test pool indexing and submission packaging.
def create_submission(model,data,store):
    queries=data['test_queries'];candidates=data['test_candidates'];qids=list(queries);cids=sorted(candidates)
    q=embed_records(model,[queries[i] for i in qids],store,'Test queries')
    c=embed_records(model,[candidates[i] for i in cids],store,'Test candidates')
    pred=rank_embeddings(qids,q,cids,c,TOP_K)
    validate_predictions(pred,queries,candidates)
    atomic_json(pred,PREDICTION_PATH)
    tmp=SUBMISSION_PATH.with_suffix('.tmp')
    with zipfile.ZipFile(tmp,'w',compression=zipfile.ZIP_DEFLATED) as z:
        z.write(PREDICTION_PATH,arcname='prediction.json')
    os.replace(tmp,SUBMISSION_PATH)
    with zipfile.ZipFile(SUBMISSION_PATH) as z:
        if z.namelist()!=['prediction.json']: raise ValueError('ZIP must contain only root-level prediction.json.')
        loaded=json.loads(z.read('prediction.json'))
        validate_predictions(loaded,queries,candidates)
        if loaded!=pred: raise ValueError('ZIP JSON differs from the validated predictions.')
    # Convenient Kaggle Output copy; per-run artifacts remain in EXP_DIR.
    import shutil
    output_zip=OUTPUT_ROOT/f'submission_{EXPERIMENT}.zip'
    shutil.copy2(SUBMISSION_PATH,output_zip)
    print('SUBMISSION READY:',output_zip,flush=True)
    return {'zip_path':str(output_zip),'run_zip_path':str(SUBMISSION_PATH),
            'prediction_path':str(PREDICTION_PATH),'queries':len(pred),
            'candidates_per_query':7,'zip_sha256':sha256_file(output_zip)}


## Section 9 · Run the pipeline and log results

This final cell performs the work. Training happens in `fit_stage()` in Section 7; test indexing and ZIP creation happen in `create_submission()` in Section 8. Logs show each stage explicitly.

Outputs include `best_<EXP>.pt`, `data_audit.json`, `results_<EXP>.json`, `prediction.json`, and `submission_<EXP>.zip`. KD adds a trained teacher checkpoint and its cache. Results record the actual measured budget, successful optimizer updates, all dev metrics, data/checkpoint/tokenizer fingerprints, and the validated submission path. A failed run writes its error to the results file and re-raises; it does not manufacture predictions.


In [49]:
# Section 9 — orchestration and durable experiment results.
def runtime_checks():
    validate_configuration()
    if CHUNKS_PER_FORWARD!=1: raise ValueError('This memory-bounded implementation uses one unpadded chunk per forward.')
    if EXCLUDE_SELF_MATCH: raise ValueError('No self-exclusion rule is specified by the task; keep EXCLUDE_SELF_MATCH=False.')
    if not USE_IN_BATCH_NEGATIVES: raise ValueError('This implementation uses a shared in-batch candidate matrix.')
    if HARD_NEGATIVE_SOURCE!='train_only_bm25_then_dense': raise ValueError('Use the implemented train-only miner.')
    if MULTI_POSITIVE_LOSS!='positive_logsumexp': raise ValueError('Unsupported positive loss.')
    if STUDENT_LAYER_POLICY!='odd_indices': raise ValueError('Implemented KD pruning retains odd zero-based indices.')
    if TEACHER_METHOD!='full': raise ValueError('KD teacher uses full fine-tuning.')
    if KD_SIGNAL!='similarity_distribution': raise ValueError('KD uses similarity distributions.')
    if CHUNK_OVERLAP<0: raise ValueError('Negative chunk overlap.')
    if EMPTY_TRAIN_GOLD_POLICY!='skip_and_audit' or INVALID_GOLD_ID_POLICY!='error' or EMPTY_DOCUMENT_POLICY!='error':
        raise ValueError('Keep the documented data-integrity policies.')
    if TOKEN_MEMORY_CACHE_DOCS<1 or INFERENCE_BENCHMARK_DOCS<1: raise ValueError('Invalid cache/benchmark size.')
    if GRAD_CKPT: raise ValueError('Use RETRIEVAL_GRAD_CKPT; original forward checkpoint paths are not used here.')
    if DEVICE.type!='cuda' and not ALLOW_CPU_PIPELINE:
        raise RuntimeError('Enable Kaggle GPU T4. CPU is supported only for explicit smoke tests.')

def effective_configuration():
    names = ['SEED','CKPT_PATH','TOK_PATH','HF_DATASET_REPO','HF_REVISION','LOCAL_DATA_DIR',
        'CHUNK_TOK','CHUNK_OVERLAP','MAX_CHUNKS','CHUNK_POLICY','APPEND_EOS_PER_CHUNK','TOKEN_POOL','CHUNK_POOL',
        'EMBED_DIM','HEAD_DROPOUT','RANK','LORA_ALPHA','ADAPTER_DIM','ADAPTER_DROPOUT','SOFT_PROMPT_TOKENS',
        'KV_PREFIX_TOKENS','NF4_BLOCK_SIZE','INFO_NCE_TEMPERATURE','CONTRASTIVE_BATCH_QUERIES',
        'POSITIVES_PER_QUERY','HARD_NEGATIVES_PER_QUERY','RANDOM_NEGATIVES_PER_QUERY','HARD_MINING_TOPK',
        'HARD_REFRESH_EPOCHS','GRAD_ACCUM','USE_GRAD_CACHE','RETRIEVAL_GRAD_CKPT','EPOCHS',
        'TRAIN_QUERIES_PER_EPOCH','MAX_DEV_QUERIES','TOTAL_GPU_MINUTES','BUDGET_SAFETY_FRACTION',
        'BENCHMARK_MARGIN','WEIGHT_DECAY','CLIP_NORM','WARMUP_FRACTION','KD_ALPHA','KD_TEMPERATURE',
        'TEACHER_POLICY','TEACHER_BUDGET_FRACTION','STRICT_STRING_IDS','TOP_K']
    return {name:globals()[name] for name in names}

def main():
    global TOKENIZER,TOKEN_STORE,DATA_FINGERPRINTS
    runtime_checks();started=time.monotonic();EXP_DIR.mkdir(parents=True,exist_ok=True)
    # A conservative wall-clock budget includes asset loading and CPU preprocessing too.
    target_seconds=TOTAL_GPU_MINUTES*60
    deadline=started+target_seconds*(1-BUDGET_SAFETY_FRACTION)
    results={'status':'running','experiment':EXPERIMENT,'backbone':BACKBONE,
             'preset':asdict(EXP),'configuration':effective_configuration(),'budget_minutes':TOTAL_GPU_MINUTES,
             'limitations':['Runtime estimates are measured, not a guarantee of a hardware deadline.',
                            'NF4 uses packed base weights but transient dense layer dequantization; no fused kernels, double quantization or paged optimizer.',
                            'Hybrid attention KV is not constant-memory; PCR uses bounded parallel SSD windows.']}
    try:
        print('[1/8] Loading trained Phase-1 checkpoint',flush=True)
        base,TOKENIZER,meta=load_phase1();results['phase1']=meta
        print('[2/8] Reading official PCR splits and tokenizing',flush=True)
        data,audit=load_data();results['data_audit']=audit
        DATA_FINGERPRINTS=audit['file_sha256']
        TOKEN_STORE=TokenStore(TOKENIZER,meta['tokenizer_sha256']);prepare_tokens(data,TOKEN_STORE)
        hard=initial_hard_negatives(data)
        dev=data['dev_queries']
        if MAX_DEV_QUERIES is not None and len(dev)>MAX_DEV_QUERIES:
            chosen=sorted(dev);random.Random(SEED).shuffle(chosen);dev={i:dev[i] for i in chosen[:MAX_DEV_QUERIES]}
        results['dev_evaluation']={'queries':len(dev),'full_split':len(dev)==len(data['dev_queries']),
                                   'candidate_pool_complete':True}
        cache=None
        print('[3/8] Constructing retrieval model and measuring throughput',flush=True)
        if USE_KD:
            teacher_preset=Preset('full',2e-5,3e-4)
            teacher=RetrievalModel(base,teacher_preset).to(DEVICE);del base
            teacher_path=EXP_DIR/f'teacher_{EXPERIMENT}.pt'
            if TEACHER_POLICY=='load_phase7_teacher':
                ck=torch.load(TEACHER_RETRIEVAL_CKPT,map_location='cpu',weights_only=False)
                if ck.get('format')!='cs781_pcr_v1' or ck.get('backbone')!=BACKBONE or ck['preset']['method']!='full':
                    raise ValueError('Provide a full-depth, full-FT PCR teacher checkpoint from this notebook family.')
                if ck['cfg']!=asdict(teacher.backbone.c) or ck['embedding_dim']!=EMBED_DIM or ck['chunk_pool']!=CHUNK_POOL:
                    raise ValueError('Teacher architecture/head settings differ.')
                if ck['assets']['tokenizer_sha256']!=meta['tokenizer_sha256'] or ck['token_signature']!=TOKEN_STORE.signature:
                    raise ValueError('Teacher tokenizer or chunking settings differ.')
                if any(ck.get('data_fingerprints',{}).get(k)!=DATA_FINGERPRINTS[k] for k in ('train_queries','train_candidates')):
                    raise ValueError('Teacher training-data fingerprints differ or are missing.')
                teacher.load_state_dict(ck['model'],strict=True);del ck
                save_retrieval(teacher,teacher_path,meta,extra={'loaded_from':TEACHER_RETRIEVAL_CKPT})
            inf_sec=benchmark_inference(teacher,data,TOKEN_STORE)
            # Reserve future student indexing conservatively at full-teacher speed.
            reserve=reserve_inference(data,dev,TOKEN_STORE,inf_sec,EPOCHS,kd_cache=True)
            teacher_bench=benchmark_training(teacher,data,TOKEN_STORE,hard)
            train_available=deadline-time.monotonic()-reserve['total']-STUDENT_BENCHMARK_RESERVE_SECONDS
            if train_available<=0: raise BudgetError(f'Budget too small after inference reserve: {reserve}')
            if TEACHER_POLICY=='train_then_cache':
                print('[4/8] Training full-depth PCR teacher',flush=True)
                teacher_train=fit_stage(teacher,data,None,TOKEN_STORE,hard,teacher_preset,
                    train_available*TEACHER_BUDGET_FRACTION,teacher_bench,meta,epochs=1,checkpoint_path=teacher_path)
                results['teacher_training']=teacher_train
            print('[4/8] Caching teacher embeddings on train only',flush=True)
            teacher.requires_grad_(False)
            cache=cache_teacher(teacher,data,TOKEN_STORE,meta,teacher_path)
            teacher.cpu();gc.collect()
            if DEVICE.type=='cuda':torch.cuda.empty_cache()
            model,keep=make_student(teacher,EXP);results['teacher_parameters']=model_counts(teacher)
            results['student_retained_indices']=keep;del teacher
            model=model.to(DEVICE)
        else:
            model=RetrievalModel(base,EXP).to(DEVICE);del base
        print('Retrieval model:',model_counts(model),'| adapted modules:',model.adapted_modules,flush=True)
        results['model']=model_counts(model)
        benchmark=benchmark_training(model,data,TOKEN_STORE,hard)
        inf_sec=benchmark_inference(model,data,TOKEN_STORE)
        reserve=reserve_inference(data,dev,TOKEN_STORE,inf_sec,EPOCHS)
        available=deadline-time.monotonic()-reserve['total']
        results['benchmark']=benchmark;results['inference_seconds_per_chunk']=inf_sec;results['reserved_seconds']=reserve
        if available<=0: raise BudgetError(f'Budget cannot fit required indexing/evaluation. Reserve={reserve}; increase TOTAL_GPU_MINUTES or lower MAX_CHUNKS.')
        print('[5/8] Training Phase-7 retrieval model | training seconds available:',round(available),flush=True)
        results['training']=fit_stage(model,data,dev,TOKEN_STORE,hard,EXP,available,benchmark,meta,
            cache=cache,epochs=EPOCHS,checkpoint_path=BEST_CHECKPOINT_PATH,global_deadline=deadline,reserve=reserve)
        best=torch.load(BEST_CHECKPOINT_PATH,map_location='cpu',weights_only=False)
        results['best_dev']=best['metrics'];results['best_checkpoint']=str(BEST_CHECKPOINT_PATH);del best
        print('[6/8] Indexing complete test candidate pool and ranking',flush=True)
        results['submission']=create_submission(model,data,TOKEN_STORE)
        print('[7/8] Submission keys, seven unique IDs/query, candidate membership and ZIP root validated',flush=True)
        results['status']='completed';results['elapsed_seconds']=time.monotonic()-started
        results['within_configured_budget']=results['elapsed_seconds']<=target_seconds
        if not results['within_configured_budget']:
            print('Actual runtime exceeded the configured estimate; complete submission retained. Inspect results timing.')
        atomic_json(results,RESULTS_PATH)
        print('[8/8] Results saved:',RESULTS_PATH,flush=True)
        try:
            from IPython.display import display,FileLink
            display(FileLink(results['submission']['zip_path']))
        except ImportError:pass
        return results
    except Exception as exc:
        results.update(status='failed',error_type=type(exc).__name__,error=str(exc),elapsed_seconds=time.monotonic()-started)
        atomic_json(results,RESULTS_PATH)
        raise

# Run all cells after setting checkpoint/tokenizer paths, preset and official data access in Section 1.
if RUN_PIPELINE:
    RESULTS=main()


[1/8] Loading trained Phase-1 checkpoint
LOADED PHASE-1 WEIGHTS: /kaggle/input/models/ashishkj93/phase-1-decoder-bundle/pytorch/default/1/best.pt | strict=True
Tokenizer: /kaggle/input/models/ashishkj93/phase-1-decoder-bundle/pytorch/default/1/bpe_merges.json | SHA256: 8f2503677817a580790dea20ef24ffa1ec710675c54d2a7eaf9e91e3435a1c67
Checkpoint configuration: {'vocab_size': 32000, 'n_ctx': 1280, 'n_layer': 12, 'n_head': 12, 'n_kv_head': 4, 'n_embd': 768, 'n_ffn': 2816, 'rope_base': 10000.0}
Phase-1 parameters: 121,326,336 | step: 1076
[2/8] Reading official PCR splits and tokenizing
train: 827 queries; 4320 candidates
dev: 118 queries; 1023 candidates
test: 237 queries; 1727 candidates
Tokenized 1000 documents in current split/pool
Tokenized 2000 documents in current split/pool
Tokenized 3000 documents in current split/pool
Tokenized 4000 documents in current split/pool
Tokenized 1000 documents in current split/pool
Tokenized 1000 documents in current split/pool
Tokenization/cache secon

/kaggle/working/submission_D2_qlora.zip

In [50]:
# Final cell — attempt to download this experiment's submission
import base64
import html
import os
import uuid
import zipfile
from pathlib import Path
from IPython.display import display, HTML, FileLink

submission = Path("/kaggle/working") / f"submission_{EXPERIMENT}.zip"

if not submission.is_file():
    raise FileNotFoundError(
        f"Submission is not ready: {submission}\n"
        "Let the pipeline finish successfully before running this cell."
    )

# Confirm this is the expected completed submission archive.
with zipfile.ZipFile(submission) as archive:
    if archive.namelist() != ["prediction.json"]:
        raise ValueError("Expected only prediction.json at the ZIP root.")
    if archive.testzip() is not None:
        raise ValueError("The submission ZIP failed its integrity check.")

payload = base64.b64encode(submission.read_bytes()).decode("ascii")
filename = html.escape(submission.name, quote=True)
link_id = "download_" + uuid.uuid4().hex

display(HTML(f"""
<p>Submission ready. If the download does not start, click below:</p>
<a id="{link_id}"
   href="data:application/zip;base64,{payload}"
   download="{filename}">
   Download {filename}
</a>
<script>
    document.getElementById("{link_id}").click();
</script>
"""))

print("Alternative file link:")
display(FileLink(os.path.relpath(submission, Path.cwd())))

Alternative file link:


/kaggle/working/submission_D2_qlora.zip

### Provenance and verification limits

```json
{
  "source_notebook": "cs781-phase5-lsi-decoder.ipynb",
  "tokenizer_source_cell": 4,
  "backbone_source_cell": 5,
  "tokenizer_source_sha256": "3be37306135fc902156959dc4c3f3002b276828c94f0ca1159e587461e5173a6",
  "backbone_source_sha256": "84e068e4e0e63e4211d6277099967937277aefa2e89a5d86bbd41d35882675dc"
}
```

Official source: the supplied challenge PDF, Phase 7 (pages 44–49).

Implementation references (no model downloads): [PyTorch AMP](https://docs.pytorch.org/docs/stable/notes/amp_examples.html), [activation checkpointing](https://docs.pytorch.org/docs/stable/checkpoint.html), and [NF4 codebook](https://github.com/bitsandbytes-foundation/bitsandbytes/blob/main/bitsandbytes/functional.py).

No real Kaggle score or trained submission is included in this download. The notebook generates those using your assets when run.

### Development validation completed

Executed on CPU with PyTorch 2.15.0+cpu using explicitly synthetic small checkpoints and official-schema synthetic data:

- All eight presets in this notebook passed finite-gradient checks; chunk-feature replay matched ordinary backpropagation, including adapter/dropout RNG replay.
- LORMA matched the explicit multiplicative matrix formula, including a frozen nonzero bias.
- Packed NF4 storage and the frozen quantized layer's input gradients passed checks.
- Strict Phase-1 loading, half-depth student construction, train-only mining, teacher fitting/caching, optimizer updates, best-checkpoint reload and dev evaluation were exercised.
- Two complete pipelines per architecture (LORMA and KD) produced validated seven-ID predictions and ZIPs containing only root-level prediction.json.
- Top-seven tie handling, full gold-set micro metrics, rejection of invalid predictions, and original tokenizer/backbone source equality passed.

These are software checks, not competition performance results. Real Phase-1 checkpoints, authenticated official PCR data and CUDA/T4 FP16 execution have not been tested here. Those checks run on your Kaggle session; no real submission or model score is claimed in this file.
